# Analyses statistiques NLP

***Structure du fichier***

- (##) Partie thématique
- (###) (1) Analyse synchronique puis en (2) Analyse diachronique (évolution)
- (####) Statistiques générales 
- (####) Tableau et graphique en VA
- (####) Tableau et graphique en %
- (####) Tableau et graphique % et VA

==> ***Nom des variables*** :
- colonne_condition = variable "République" analysée 
- colonne_groupe/député/ = variable 

***Règles d'or : Comparer systématiquement statistiques générales, proportions en % et valeurs absolues sur les 2 df afin de pouvoir distinguer ce qui relève du sur ou du sous-investissement.***

*À garder en tête au moment de l'analyse et interprétation des résultats, l'analyse en % dépend de notre unité de mesure (nombre de prises de paroles avec ou sans interruption, phrases). On ne peut pas mesurer en durée de l'intervention ou nombre de mots pour l'intervention donc il ne s'agit pas à proprement parlé d'un % en termes de temps de parole (= un artéfact statistique dont il serait intéressant de comparer les mesures).*

==> Pour le futur croiser aussi analyse avec données par phrases (utiliser spacy)

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv(
    "../data/interim/df_repu_proportion.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [3]:
df_regroup = pd.read_csv(
    "../data/interim/df_regroup_repu_proportion.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [4]:
import datetime
import locale

# Active la locale française (nécessaire pour le format)
locale.setlocale(locale.LC_TIME, "fr_FR.UTF-8")

'fr_FR.UTF-8'

In [8]:
df["dateSeance_ts"] = pd.to_datetime(df["dateSeanceJour"], format="%A %d %B %Y")
df["dateSeance_day"] = df["dateSeance_ts"].dt.normalize()  

df_regroup["dateSeance_ts"] = pd.to_datetime(df_regroup["dateSeanceJour"], format="%A %d %B %Y")
df_regroup["dateSeance_day"] = df_regroup["dateSeance_ts"].dt.normalize()  

In [5]:
# Correspondances couleurs et affiliations
couleurs_groupes = {
    "LFI": "#B71C1C",  # rouge foncé
    "GDR": "#ea193a",  # rouge vif
    "ECO": "#1FAE70",  # vert
    "SOC-A": "#F05FA8",# rose
    "REN": "#FFD83E",  # jaune 
    "DEM": "#F57C00",  # orange 
    "HOR": "#6994D6",  # bleu gris
    "LIOT": "#C075C0",  # violet clair repéré sur Wikipédia/instituts de sondages
    "AGIR-E": "#2D769E",  # jaune turquoise
    "UDI": "#439FD1",  # bleu clair
    "LR": "#0D47A1",   # bleu foncé
    "RN": "#412302", # marron foncé ou noir "#000000" 
    # Option Générique par défaut
    "Autres": "#9E9E9E"}

couleurs_bloc = {
    "Gauche": "#A92424", # rouge
    "Centre": "#D59629", # orange
    "Droite": "#1F57AC",   # bleu
    # Valeur par défaut si un parti n'est pas défini
    "Autres": "rgb(160, 160, 160)"
}

## Analyses générales temporelles

### Statistiques générales 

In [ ]:
def statistiques_générales_temporelles(df, periode="semaine", colonne_condition="repu_match_valide"):

    # Définir la granularité
    if periode == "semaine":
        df["periode"] = (
            df["dateSeance_day"].dt.isocalendar().year.astype(str)
            + "-W"
            + df["dateSeance_day"].dt.isocalendar().week.astype(str)
        )
    elif periode == "mois":
        df["periode"] = df["dateSeance_day"].dt.to_period("M").astype(str)
    elif periode == "annee":
        df["periode"] = df["dateSeance_day"].dt.year.astype(str)
    elif periode == "global":
        df["periode"] = "Global"


    # Compter occurrences par période (proportion et True)
    df_counts = (
        df.groupby("periode")[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Proportion de True
    df_counts["proportion_true"] = df_counts["true_mentions"] / df_counts["total_mentions"] 

    # Statistiques 
    stats = {
        "moyenne_proportion": df_counts["proportion_true"].mean(),
        "médiane_proportion": df_counts["proportion_true"].median(),
        "moyenne_valeur": df_counts["true_mentions"].mean(),
        "médiane_valeur": df_counts["true_mentions"].median(),
        "deciles_proportion": df_counts["proportion_true"].quantile([i/100 for i in range(90, 100)])*100,
        "deciles_valeur": df_counts["true_mentions"].quantile([i/100 for i in range(90, 100)])
    }
    
    return stats

### Par ans

#### Statistiques

In [ ]:
# --- Par année ---
stats_annee = statistiques_générales_temporelles(df, periode="annee")
print("Statistiques Par Année :")
print(f"Moyenne (VA) : {stats_annee['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_annee['médiane_valeur']:.2f}")
print(f"Déciles (VA)  : {stats_annee['deciles_valeur']}")
print(f"Moyenne (%)  : {stats_annee['moyenne_proportion']:.2%}")
print(f"Médiane (%) : {stats_annee['médiane_proportion']:.2f}")
print(f"Déciles (%)  : {stats_annee['deciles_proportion']}")

#### Tableau(x) et graphique(s)

In [ ]:
# Grouper par année
df_yearly = (
    df.groupby(df["dateSeance_day"].dt.year)["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Garder True uniquement
df_yearly_true = df_yearly[df_yearly["repu_match_valide"] == True]

# Graphique
fig_yearly = px.bar(
    df_yearly_true,
    x="dateSeance_day",
    y="proportion",
    title="Évolution annuelle de l'investissement en proportion de la famille du mot 'République' (B)",
    labels={"proportion": "% des occurences annualisées", "dateSeance_day": "Année"},
    template="plotly_white",
)

fig_yearly.show()
table = df_yearly_true.sort_values("proportion", ascending=False).head(10)
table

### Par mois

#### Statistiques 

In [ ]:

# --- Par mois ---
stats_mois = statistiques_générales_temporelles(df, periode="mois")
print("Statistiques Par mois :")
print(f"Moyenne (VA) : {stats_mois['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_mois['médiane_valeur']:.2f}")
print(f"Déciles (VA)  : {stats_mois['deciles_valeur']}")
print(f"Moyenne (%)  : {stats_mois['moyenne_proportion']:.2%}")
print(f"Médiane (%) : {stats_mois['médiane_proportion']:.2f}")
print(f"Déciles (%)  : {stats_mois['deciles_proportion']}")

#### Tableau(x) et graphique(s)

In [ ]:
# Grouper par mois
df_monthly = (
    df.groupby(df["dateSeance_day"].dt.to_period("M"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_monthly["dateSeance_day"] = df_monthly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_monthly_true = df_monthly[df_monthly["repu_match_valide"] == True]

# Graphique
fig_monthly = px.bar(
    df_monthly_true,
    x="dateSeance_day",
    y="proportion",
    title="Évolution mensuelle de l'investissement en valeur absolue de la famille du mot 'République' (B)",
    labels={"proportion": "% des occurences mensualisées", "dateSeance_day": "Date"},
    template="plotly_white",
)

fig_monthly.show()
# aficher les 10 mois les plus fréquents sous forme de tableau
table = df_monthly_true.sort_values("proportion", ascending=False).head(10)
table


### Par semaines

#### Statistiques

In [ ]:

# --- Par semaine ---
stats_sem = statistiques_générales_temporelles(df, periode="semaine")
print("Statistiques Par semaines :")
print(f"Moyenne (VA) : {stats_sem['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_sem['médiane_valeur']:.2f}")
print(f"Déciles (VA)  : {stats_sem['deciles_valeur']}")
print(f"Moyenne (%)  : {stats_sem['moyenne_proportion']:.2%}")
print(f"Médiane (%) : {stats_sem['médiane_proportion']:.2f}")
print(f"Déciles (%)  : {stats_sem['deciles_proportion']}")

#### Tableau(x) et graphique(s)

In [ ]:
# Grouper par semaine
df_weekly = (
    df.groupby(df["dateSeance_day"].dt.to_period("W"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_weekly["dateSeance_day"] = df_weekly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_weekly_true = df_weekly[df_weekly["repu_match_valide"] == True]

# Graphique
fig_weekly = px.bar(
    df_weekly_true,
    x="dateSeance_day",
    y="proportion",
    title="Évolution hebdomadaire de l'investissement en valeur absolue de la famille du mot 'République' (B)",
    labels={"proportion": "% des occurences hebdomaire", "dateSeance_day": "Semaine"},
    template="plotly_white",
)

fig_weekly.show()

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_weekly_true.sort_values("proportion", ascending=False).head(20)
table


In [ ]:
# Compter True/False par jour
df_daily = (
    df.groupby(df["dateSeance_day"].dt.date)["repu_match_valide"]
    .value_counts(normalize=True)  # calcule directement les proportions
    .rename("proportion")
    .reset_index()
)

# Garder uniquement les "True"
df_daily_true = df_daily[df_daily["repu_match_valide"] == True]

# Graphique
fig_daily = px.line(
    df_daily_true,
    x="dateSeance_day",
    y="proportion",
    title="Évolution journalier de l'investissement en proportion de la famille du mot 'République' (B)",  
    labels={"proportion": "% des occurences", "dateSeance_day": "Date"},
    template="plotly_white",
)

fig_daily.show()

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_daily_true.sort_values("proportion", ascending=False).head(20)
table


**Remarques**
- *Le 3 juillet 2017, le 9 juillet 2018, le 4 mars 2024 sont parmis les 5 principales dates car parlement réuni en Congrès*

- *le 1 février 2021, le 28 juin 2021, 5 février 2021, le 23 juillet 2021, le 3 février 2021, 30 juin 2021 (et 5 avril 2023 : bilan de la loi), 1er juillet 2021, le 12 février 2021, renvoient quant à eux à la discussion du projet de loi "confortant le respect des principes de la République". **==> Suivre en détail le processus législatif de ce projet de loi car moment central !!***

- *On a aussi le 22 mars 2020, très courte séance (commencée à 18h30) sur l’urgence du covid et un hommage*

- *6 janvier 2022, 6 juin 2022, 13 mars 2018 : réforme territoriale Nouvelle-Calédonie*

- *25 janvier 2024, 8 juillet 2019 et 16 janvier 2020 : accords internationaux avec présence d'expressions comme "gouvernement de la République française", de pays sous forme adjectivable (ex : "république arménienne") ou avec république en miniscule --> moins présent maintenant que exclus*

- *11 février 2019 sur "l'école de la confiance"*
- *12 et 13 juillet 2018 sur le  projet de loi constitutionnelle pour une Démocratie plus représentative, responsable et efficace*

#### Par période sur corpus annualisé 

In [ ]:
# Grouper par mois
df_period = (
    df.groupby(df["dateSeance_day"].dt.to_period("M"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_period["dateSeance_day"] = df_period["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_period_true = df_period[df_period["repu_match_valide"] == True]

# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_period_period = df_period_true[df_period_true["dateSeance_day"].dt.year == annee]

# Graphique
fig_period_period = px.bar(
    df_period_period,
    x="dateSeance_day",
    y="proportion",
    title="Proportion occurrences de la 'République' par mois en 2021",
    labels={"proportion": "% des occurences mensualisées", "dateSeance_day": "Mois"},
    template="plotly_white",
)

fig_period_period.show()

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_period_true.sort_values("proportion", ascending=False).head(50)
table

### Graphiques finaux 

In [ ]:
# Définir les paramètres
colonne_condition = "repu_match_valide"
periode = "W"  # "ME" = mois, "WE" = semaine, "YS" = année.

# S'assurer que la colonne date est bien en datetime
df["dateSeance_day"] = pd.to_datetime(df["dateSeance_day"])

# Agréger les données par mois
df_monthly = (
    df
    .resample(periode, on="dateSeance_day")[colonne_condition]
    .agg(
        total_mentions="count",
        true_mentions=lambda x: (x == True).sum()
    )
    .reset_index()
)

# Calculer la proportion
df_monthly["proportion_true"] = df_monthly["true_mentions"] / df_monthly["total_mentions"]

# Créer la figure avec Plotly Graph Objects
fig = go.Figure()

# Barres : volume total
fig.add_trace(go.Bar(
    x=df_monthly["dateSeance_day"],
    y=df_monthly["true_mentions"],
    name="Nombre d'interventions avec FDM 'République'",
    marker_color="rgba(99, 110, 250, 0.6)",
    yaxis="y1"
))

# Courbe : proportion
fig.add_trace(go.Scatter(
    x=df_monthly["dateSeance_day"],
    y=df_monthly["proportion_true"] * 100,  # en pourcentage
    name="Proportion d'intervention en % avec FDM 'République'",
    mode="lines+markers",
    line=dict(color="rgba(239, 85, 59, 0.8)", width=3),
    yaxis="y2"
))

# Mise en forme du graphique
fig.update_layout(
    title=f"Évolution hebdomadaire de l'investissement de la famille du mot 'République' (A)",
    xaxis=dict(title="Mois"),
    yaxis=dict(
        title="Valeurs absolue",
        showgrid=False
    ),
    yaxis2=dict(
        title="Proportion en %",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(x=0.65, y=1.14, bgcolor="rgba(255,255,255,0.7)"),
    template="plotly_white",
    bargap=0.2
)
fig.update_xaxes(
    dtick="M12",  # un tick par an
    tickformat="%Y",
    ticklabelmode="period"  # place les labels entre les périodes
)

fig.show()


## Par groupes 

### Analyse synchronique  

#### Statistiques 

In [ ]:
def statistiques_générales_groupes(df, periode="semaine", colonne_condition="repu_match_valide"):

    # Définir la granularité
    if periode == "semaine":
        df["periode"] = (
            df["dateSeance_day"].dt.isocalendar().year.astype(str)
            + "-W"
            + df["dateSeance_day"].dt.isocalendar().week.astype(str)
        )
    elif periode == "mois":
        df["periode"] = df["dateSeance_day"].dt.to_period("M").astype(str)
    elif periode == "annee":
        df["periode"] = df["dateSeance_day"].dt.year.astype(str)
    elif periode == "global":
        df["periode"] = "Global"


    # Compter occurrences par période (proportion et True)
    df_counts = (
        df.groupby(["groupe_députés_affiliation", "periode"])[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Proportion de True
    df_counts["proportion_true"] = df_counts["true_mentions"] / df_counts["total_mentions"] 

    # Statistiques 
    stats = {
        "moyenne_proportion": df_counts["proportion_true"].mean(),
        "médiane_proportion": df_counts["proportion_true"].median(),
        "moyenne_valeur": df_counts["true_mentions"].mean(),
        "médiane_valeur": df_counts["true_mentions"].median(),
        "deciles_proportion": df_counts["proportion_true"].quantile([i/100 for i in range(1, 100)])*100,
        "deciles_valeur": df_counts["true_mentions"].quantile([i/100 for i in range(1, 100)])
    }
    
    return stats

In [ ]:


# Sur toute la période 
stats_global = statistiques_générales_groupes(df, periode="global") # remplacer df par df_regroup pour avoir le corpus B
print("Statistiques sur 2017-2024")
print(f"Moyenne (VA) : {stats_global['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_global['médiane_valeur']:.2f}")
print(f"Déciles (VA)  : {stats_global['deciles_valeur']}")
print(f"Moyenne (%)  : {stats_global['moyenne_proportion']:.2%}")
print(f"Médiane (%) : {stats_global['médiane_proportion']:.2f}")
print(f"Déciles (%)  : {stats_global['deciles_proportion']}")

# --- Par année ---
# stats_annee = statistiques_générales_groupes(df, periode="annee")
#print("Par Année :")
#print(f"Moyenne (VA) : {stats_annee['moyenne_valeur']:.2f}")
#print(f"Médiane (VA) : {stats_annee['médiane_valeur']:.2f}")
#print(f"Déciles (VA)  : {stats_annee['deciles_valeur']}")
#print(f"Moyenne (%)  : {stats_annee['moyenne_proportion']:.2%}")
#print(f"Médiane (%) : {stats_annee['médiane_proportion']:.2f}")
#print(f"Déciles (%)  : {stats_annee['deciles_proportion']}")

# --- Par mois ---
# stats_mois = statistiques_générales_groupes(df, periode="mois")
#print("Par Mois :")
#print(f"Moyenne (VA) : {stats_mois['moyenne_valeur']:.2f}")
#print(f"Médiane (VA) : {stats_mois['médiane_valeur']:.2f}")
#print(f"Déciles (VA)  : {stats_mois['deciles_valeur']}")
#print(f"Moyenne (%)  : {stats_mois['moyenne_proportion']:.2%}")
#print(f"Médiane (%) : {stats_mois['médiane_proportion']:.2f}")
#print(f"Déciles (%)  : {stats_mois['deciles_proportion']}")

# --- Par semaine ---
#stats_sem = statistiques_générales_groupes(df, periode="semaine")
#print("Par Semaine :")
#print(f"Moyenne (VA) : {stats_sem['moyenne_valeur']:.2f}")
#print(f"Médiane (VA) : {stats_sem['médiane_valeur']:.2f}")
#print(f"Déciles (VA)  : {stats_sem['deciles_valeur']}")
#print(f"Moyenne (%)  : {stats_sem['moyenne_proportion']:.2%}")
#print(f"Médiane (%) : {stats_sem['médiane_proportion']:.2f}")
#print(f"Déciles (%)  : {stats_sem['deciles_proportion']}")


#### En VA (à faire)

In [ ]:
# Sous format fonction et nommer diachronique_groupes_VA

#### En proportion (à coloriser)

In [ ]:
# Sous format fonction et nommer diachronique_groupes_%

In [ ]:
# Compter le nombre de fois où chaque groupe parlementaire dit "République"
counts = (
    df.groupby("groupe_députés_affiliation")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 250]

# Trier par proportion décroissante et garder les 40 premiers
df_toppartis = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig_toppartis = px.bar(
    df_toppartis,
    x="groupe_députés_affiliation",
    y="proportion_true",
    title="Proportions des occurrences de la 'République' par groupe parlementaire (<250 occurrences)",
    labels={"groupe_députés_affiliation": "Groupe parlementaire", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_toppartis.update_layout(xaxis_tickangle=-45)

fig_toppartis.show()
df_toppartis

#### En proportion + VA 

In [ ]:
# Sous format fonction et nommer diachronique_groupes_total

In [ ]:
# Paramètres
colonne_condition = "repu_match_valide"
colonne_groupe = "groupe_députés_affiliation"

# Compter le nombre de fois où chaque groupe parlementaire dit "République"
df_groupes = (
    df.groupby(colonne_groupe)[colonne_condition]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calculer la proportion
df_groupes["proportion_true"] = df_groupes["true_mentions"] / df_groupes["total_mentions"] * 100  # en %

# Filtrage des 10 premiers (en % ou en valeur absolue)
df_groupes = df_groupes.sort_values("true_mentions", ascending=False).head(10).reset_index(drop=True)


# Créer la figure
fig = go.Figure()

# Nombre d'occurrences de la "République"
fig.add_trace(go.Bar(
    x=df_groupes[colonne_groupe],
    y=df_groupes["true_mentions"],
    name="Nombre d'interventions avec FDM 'République'",
    marker_color="rgba(99, 110, 250, 0.6)",
    yaxis="y1"
))

# Proportion des occurrences de la "République"
fig.add_trace(go.Scatter(
    x=df_groupes[colonne_groupe],
    y=df_groupes["proportion_true"],
    name="Proportion d'intervention en % avec FDM 'République'",
    mode="markers", 
    marker=dict(color="rgba(239, 85, 59, 0.9)", size=10, symbol="circle"),
    yaxis="y2"
))

# Mise en forme
fig.update_layout(
    title="Investissement de la famille du mot 'République' des dix principaux groupes parlementaires (A)",
    xaxis=dict(title="Groupe parlementaire"),
    yaxis=dict(
        title="Valeur absolue",
        showgrid=False
    ),
    yaxis2=dict(
        title="Proportion en %",
        overlaying="y",
        side="right",
        showgrid=False,
    ),
    legend=dict(x=0.63, y=-0.28, bgcolor="rgba(255,255,255,0.7)"),
    template="plotly_white",
    bargap=0.25
)

# Afficher la figure
fig.show()
df_groupes


In [ ]:
# Graphique avec couleurs spécifiques par parti

# Agréger les données par parti
colonne_condition = "repu_match_valide"
colonne_groupe = "groupe_députés_affiliation"

df_groupes = (
    df.groupby(colonne_groupe)[colonne_condition]
    .agg(
        total_mentions="count",
        true_mentions=lambda x: (x == True).sum()
    )
    .reset_index()
)

df_groupes["proportion_true"] = df_groupes["true_mentions"] / df_groupes["total_mentions"] * 100

# Si certains partis ne sont pas dans ton dictionnaire, leur attribuer "Autres"
df_groupes["couleur"] = df_groupes[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes["Autres"])

# Trier les groupes par volume
df_groupes = df_groupes.sort_values("true_mentions", ascending=False).head(10).reset_index(drop=True)

# Créer la figure
fig = go.Figure()

# Barres : volume total (couleur spécifique par parti)
for _, row in df_groupes.iterrows():
    fig.add_trace(go.Bar(
        x=[row[colonne_groupe]],
        y=[row["true_mentions"]],
        name=row[colonne_groupe],
        marker_color=row["couleur"],
        yaxis="y1",
        showlegend=False  # On affiche la légende séparément si besoin
    ))

# Points : proportion
fig.add_trace(go.Scatter(
    x=df_groupes[colonne_groupe],
    y=df_groupes["proportion_true"],
    name="Proportion d'intervention en % avec FDM 'République'",
    mode="markers",
    marker=dict(color="black", size=10, symbol="circle"),
    yaxis="y2"
))

# Mise en page
fig.update_layout(
    title="Investissement de la famille du mot 'République' des dix principaux groupes parlementaires (A)",
    xaxis=dict(title="Groupe parlementaire"),
    yaxis=dict(title="Valeur absolue", showgrid=False),
    yaxis2=dict(
        title="Proportion en %",
        showgrid=False, 
        overlaying="y",
        side="right",
    ),
    legend=dict(x=0.65, y=1.17, bgcolor="rgba(255,255,255,0.7)"),
    template="plotly_white",
    bargap=0.35
)

fig.show()


### Analyses diachroniques (Évolutions)

#### En valeur absolue

In [ ]:
def evolution_groupes_va(df,
                                  date_col="dateSeance_day",
                                  colonne_groupe="groupe_députés_affiliation",
                                  colonne_condition="repu_match_valide",
                                  top_n=10):
    # 1. Calcul par groupe
    counts = (
        df.groupby(colonne_groupe)[colonne_condition]
        .apply(lambda x: (x == True).sum())
        .reset_index(name="true_mentions")
    )

    # 2. Trier les groupes par volume pour ne garder que les X premiers
    top_groupes = (
        counts.sort_values("true_mentions", ascending=False)
        .head(top_n)[colonne_groupe]
        .tolist()
    )
    
    # 3. Filtrer le DF
    df_groupes = df[df[colonne_groupe].isin(top_groupes)&(df[colonne_condition] == True)].copy()

    # 4. Agrégation annuelle 
    df_groupes = (
        df_groupes.groupby([pd.Grouper(key=date_col, freq="YS"), colonne_groupe])[colonne_condition]
        .apply(lambda x: (x == True).sum())  
        .reset_index(name="true_mentions")
    )

    # 5. Extraire l'année
    df_groupes["Année"] = df_groupes[date_col].dt.year

    # 6. Construire la figure avec Graph objects
    fig = go.Figure()

    for groupe in df_groupes[colonne_groupe].unique():
        subset = df_groupes[df_groupes[colonne_groupe] == groupe]

        couleur = couleurs_groupes.get(groupe, couleurs_groupes["Autres"])

        fig.add_trace(go.Scatter(
            x=subset["Année"],
            y=subset["true_mentions"],
            mode="lines+markers",
            name=groupe,
            line=dict(color=couleur, width=2),
            marker=dict(size=6, color=couleur),
            hovertemplate="<b>%{x}</b><br>%{y} occurences<extra>" + groupe + "</extra>"
        ))

    # 7. Mise en forme du graphique 
    fig.update_layout(
        title=f"Évolution de l'investissement de la famille du mot 'République' des {top_n} principaux groupes parlementaires en VA (A)",
        xaxis=dict(
            title="Année",
            tickmode="linear",
            dtick=1,
            showgrid=True
        ),
        yaxis=dict(
            title="Valeur absolue",
            showgrid=True,
            gridcolor="#D7D6D6"
        ),
        legend=dict(title="Groupes", x=1.02, y=.95, bgcolor="rgba(255,255,255,0.8)"),
        template="plotly_white",
        hovermode="x unified",
        
        font=dict(size=12)
    )

    fig.show()


In [ ]:
evolution_groupes_va(df)

#### En proportion

In [ ]:
def evolution_groupes_proportion(df,
                                  date_col="dateSeance_day",
                                  colonne_groupe="groupe_députés_affiliation",
                                  colonne_condition="repu_match_valide",
                                  top_n=10):
    # 1. Calcul global des proportions par groupe
    counts = (
        df.groupby(colonne_groupe)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 2. Trier les groupes par volume pour ne garder que les X premiers
    top_groupes = counts.sort_values("true_mentions", ascending=False).head(top_n)[colonne_groupe].tolist()
    
    # 3. Filtrer le DF
    df_groupes = df[df[colonne_groupe].isin(top_groupes)].copy()
    
    # 4. Agrégation annuelle 
    df_groupes = (
        df_groupes.groupby([pd.Grouper(key=date_col, freq="YS"), colonne_groupe])[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 5. Calcul du pourcentage annuel 
    df_groupes["proportion_true"] = df_groupes["true_mentions"] / df_groupes["total_mentions"]
    df_groupes["Année"] = df_groupes[date_col].dt.year

    # 6. Construire la figure avec Graph objects
    fig = go.Figure()

    for groupe in df_groupes[colonne_groupe].unique():
        subset = df_groupes[df_groupes[colonne_groupe] == groupe]

        couleur = couleurs_groupes.get(groupe, couleurs_groupes["Autres"])

        fig.add_trace(go.Scatter(
            x=subset["Année"],
            y=subset["proportion_true"] * 100,
            mode="lines+markers",
            name=groupe,
            line=dict(color=couleur, width=2),
            marker=dict(size=6, color=couleur),
            hovertemplate="<b>%{x}</b><br>%{y:.1f}%<extra>" + groupe + "</extra>"
        ))

    # 7. Mise en forme du graphique 
    fig.update_layout(
        title=f"Évolution de l'investissement de la famille du mot 'République' des {top_n} principaux groupes parlementaires en % (A)",
        xaxis=dict(
            title="Année",
            tickmode="linear",
            dtick=1,
            showgrid=True
        ),
        yaxis=dict(
            title="% d'utilisation",
            showgrid=True,
            gridcolor="#D7D6D6"
        ),
        template="plotly_white",
        hovermode="x unified",
        legend=dict(title="Groupes", x=1.02, y=.95),
        font=dict(size=12)
    )

    fig.show()


In [ ]:
evolution_groupes_proportion(df)

#### Tester évolution top 5 en % et VA ???

### Par groupe

#### En valeur absolue (à faire)

In [ ]:
# diachronique_groupe_va

#### En proportion 

In [ ]:
# diachronique_groupe_proportion

#### En VA + %

In [ ]:
# diachronique_groupe_total

In [ ]:

# Définir les paramètres
colonne_groupe = "RN"
colonne_condition = "repu_match_valide"
periode = "YS"  # "M" = mois, "W" = semaine, penser à YS pour avoir début d'année 

# Filtrer les données pour le groupe
df_parti = df[df["groupe_députés_affiliation"] == colonne_groupe].copy()

# S'assurer que la colonne date est bien en datetime
df_parti["dateSeance_day"] = pd.to_datetime(df_parti["dateSeance_day"])

# Agréger les données par mois
df_period = (
    df_parti
    .resample(periode, on="dateSeance_day")[colonne_condition]
    .agg(
        total_mentions="count",
        true_mentions=lambda x: (x == True).sum()
    )
    .reset_index()
)

# Calculer la proportion
df_period["proportion_true"] = df_period["true_mentions"] / df_period["total_mentions"]

# Créer la figure avec Plotly Graph Objects
fig = go.Figure()

# Barres : volume total
fig.add_trace(go.Bar(
    x=df_period["dateSeance_day"],
    y=df_period["true_mentions"],
    name="Nombre total des occurrences",
    marker_color="rgba(99, 110, 250, 0.6)",
    yaxis="y1"
))

# Courbe : proportion
fig.add_trace(go.Scatter(
    x=df_period["dateSeance_day"],
    y=df_period["proportion_true"] * 100,  # en pourcentage
    name="Proportions des occurrences",
    mode="lines+markers",
    line=dict(color="rgba(239, 85, 59, 0.8)", width=3),
    yaxis="y2"
))

# Mise en forme du graphique
fig.update_layout(
    title=f"Évolution anuelle de l'investissement de la famille du mot 'République' à {colonne_groupe} (A)",
    xaxis=dict(title="Années"),
    yaxis=dict(
        title="Valeur absolue",
        showgrid=False
    ),
    yaxis2=dict(
        title="Proportion en %",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(x=0.77, y=1.3, bgcolor="rgba(255,255,255,0.7)"),
    template="plotly_white",
    bargap=0.2
)

fig.show()
df_period


In [ ]:
# diachronique_groupe_total_colorisé

In [ ]:
# Définir les paramètres
colonne_groupe = "REN"
colonne_condition = "repu_match_valide"
periode = "YS"  # "M" = mois, "W" = semaine, "YS" = début d'année

# Filtrage
df_parti = df[df["groupe_députés_affiliation"] == colonne_groupe].copy()

# Agrégation par période
df_period = (
    df_parti
    .resample(periode, on="dateSeance_day")[colonne_condition]
    .agg(
        total_mentions="count",
        true_mentions=lambda x: (x == True).sum()
    )
    .reset_index()
)

# Calcul de la proportion
df_period["proportion_true"] = df_period["true_mentions"] / df_period["total_mentions"]

# === 2. Couleur du parti ===
couleur_parti = couleurs_groupes.get(colonne_groupe, couleurs_groupes["Autres"])

# === 3. Création de la figure ===
fig = go.Figure()

# Barres : volume total
fig.add_trace(go.Bar(
    x=df_period["dateSeance_day"],
    y=df_period["true_mentions"],
    name="Nombre total des occurrences",
    marker_color=couleur_parti,
    opacity=0.7,
    yaxis="y1"
))

# Courbe : proportion (%)
fig.add_trace(go.Scatter(
    x=df_period["dateSeance_day"],
    y=df_period["proportion_true"] * 100,
    name="Proportion des occurrences",
    mode="lines+markers",
    line=dict(color="#000000", width=3, dash="solid"),
    marker=dict(size=8, color="#000000", line=dict(width=1, color="white")),
    yaxis="y2"
))

# === 4. Mise en forme ===
fig.update_layout(
    title=f"Évolution annuelle de l'investissement de la famille du mot 'République' au {colonne_groupe} (A)",
    xaxis=dict(title="Années"),
    yaxis=dict(
        title="Valeur absolue",
        showgrid=False
    ),
    yaxis2=dict(
        title="Proportion en %",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(
        x=0.75, y=1.2,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    template="plotly_white",
    bargap=0.2
)

fig.show()
df_period


## Par personnel politique / individuellement

### Analyse diachronique

#### Statistiques générales

In [ ]:
def diachronique_personnelpo_statistiques(df, periode="semaine", colonne_condition="repu_match_valide"):
  

    # Définir la granularité
    if periode == "semaine":
        df["periode"] = (
            df["dateSeance_day"].dt.isocalendar().year.astype(str)
            + "-W"
            + df["dateSeance_day"].dt.isocalendar().week.astype(str)
        )
    elif periode == "mois":
        df["periode"] = df["dateSeance_day"].dt.to_period("M").astype(str)
    elif periode == "annee":
        df["periode"] = df["dateSeance_day"].dt.year.astype(str)
    elif periode == "global":
        df["periode"] = "Global"


    # Compter occurrences par période et par personnel politique (proportion et True)
    df_counts = (
        df.groupby(["nom_orateur_clean", "periode"])[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Proportion de True
    df_counts["proportion_true"] = df_counts["true_mentions"] / df_counts["total_mentions"] 

    # Statistiques 
    stats = {
        "moyenne_proportion": df_counts["proportion_true"].mean(),
        "médiane_proportion": df_counts["proportion_true"].median(),
        "moyenne_valeur": df_counts["true_mentions"].mean(),
        "médiane_valeur": df_counts["true_mentions"].median(),
        "deciles_proportion": df_counts["proportion_true"].quantile([i/100 for i in range(50, 90)])*100,
        "deciles_valeur": df_counts["true_mentions"].quantile([i/100 for i in range(50, 90)])
    }
    
    return stats

In [ ]:
# Sur toute la période 
stats_global = diachronique_personnelpo_statistiques(df_regroup, periode="global") #remplacer df par df_regroup
print("Statistiques sur 2017-2024")
print(f"Moyenne (VA) : {stats_global['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_global['médiane_valeur']:.2f}")
print(f"Déciles (VA)  : {stats_global['deciles_valeur']}")
print(f"Moyenne (%)  : {stats_global['moyenne_proportion']:.2%}")
print(f"Médiane (%) : {stats_global['médiane_proportion']:.2f}")
print(f"Déciles (%)  : {stats_global['deciles_proportion']}")

# --- Par année ---
# stats_annee = diachronique_personnelpo_statistiques(df, periode="annee")
#print("Par Année :")
#print(f"Moyenne (VA) : {stats_annee['moyenne_valeur']:.2f}")
#print(f"Médiane (VA) : {stats_annee['médiane_valeur']:.2f}")
#print(f"Déciles (VA)  : {stats_annee['deciles_valeur']}")
#print(f"Moyenne (%)  : {stats_annee['moyenne_proportion']:.2%}")
#print(f"Médiane (%) : {stats_annee['médiane_proportion']:.2f}")
#print(f"Déciles (%)  : {stats_annee['deciles_proportion']}")

# --- Par mois ---
# stats_mois = diachronique_personnelpo_statistiques(df, periode="mois")
#print("Par Mois :")
#print(f"Moyenne (VA) : {stats_mois['moyenne_valeur']:.2f}")
#print(f"Médiane (VA) : {stats_mois['médiane_valeur']:.2f}")
#print(f"Déciles (VA)  : {stats_mois['deciles_valeur']}")
#print(f"Moyenne (%)  : {stats_mois['moyenne_proportion']:.2%}")
#print(f"Médiane (%) : {stats_mois['médiane_proportion']:.2f}")
#print(f"Déciles (%)  : {stats_mois['deciles_proportion']}")

# --- Par semaine ---
#stats_sem = diachronique_personnelpo_statistiques(df, periode="semaine")
#print("Par Semaine :")
#print(f"Moyenne (VA) : {stats_sem['moyenne_valeur']:.2f}")
#print(f"Médiane (VA) : {stats_sem['médiane_valeur']:.2f}")
#print(f"Déciles (VA)  : {stats_sem['deciles_valeur']}")
#print(f"Moyenne (%)  : {stats_sem['moyenne_proportion']:.2%}")
#print(f"Médiane (%) : {stats_sem['médiane_proportion']:.2f}")
#print(f"Déciles (%)  : {stats_sem['deciles_proportion']}")

#### Tableau(x) et graphique(s)

##### En valeur absolue

In [ ]:
def diachronique_personnelpo_va(df, couleurs_groupes, 
                            colonne_député="nom_orateur_clean",
                            colonne_condition="repu_match_valide",
                            colonne_groupe="groupe&gvt_affiliation",
                            top_n=10):

    # 1. Déterminer le groupe dominant de chaque personnel politique (le plus fréquent)
    groupe_dominant = (
        df.groupby([colonne_député, colonne_groupe])
        .size()
        .reset_index(name="nb_mentions")
        .sort_values(["nom_orateur_clean", "nb_mentions"], ascending=[True, False])
        .drop_duplicates(subset=colonne_député)
        .set_index(colonne_député)[colonne_groupe]
    )

    # 2. Compter le nombre de mobilisation de la FDM 'République' pour chaque personnel politique
    counts = (
        df.groupby(colonne_député)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 3. Ajouter la colonne du groupe dominant
    counts[colonne_groupe] = counts[colonne_député].map(groupe_dominant)

    # 4. Ajouter la couleur correspondante
    counts["couleur"] = counts[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes["Autres"])

    # 5. Trier les députés par le nombre de mentions "True"
    df_personnelpo_va = counts.sort_values("true_mentions", ascending=False).head(top_n).reset_index(drop=True)

    # 6. Créer le graphique
    fig = go.Figure()

    for _, row in df_personnelpo_va.iterrows():
        fig.add_trace(go.Bar(
            x=[row[colonne_député]],
            y=[row["true_mentions"]],
            name=row[colonne_député], #essayer d'enlever
            marker_color=row["couleur"],
            hovertemplate=f"<b>{row[colonne_député]}</b><br>"
                          f"Groupe : {row[colonne_député]}<br>"
                          f"Occurrences : {row['true_mentions']}",
            showlegend=False
        ))

    # 7. Mise en forme
    fig.update_layout(
        title=f"Top {top_n} du personnel politique investissant le plus la famille du mot 'République' à l'AN en valeur absolue (A)",
        xaxis=dict(title="Député·es -- Ministres", tickangle=-45),
        yaxis=dict(title="Valeur absolue", showgrid=False),
        template="plotly_white",
        bargap=0.25,
    )

    fig.show()
    return df_personnelpo_va

In [ ]:
df_personnelpo_va = diachronique_personnelpo_va(df_regroup, couleurs_groupes)
df_personnelpo_va

##### En proportion 

In [ ]:
def diachronique_personnelpo_proportion(df, couleurs_groupes, 
                            colonne_député="nom_orateur_clean",
                            colonne_condition="repu_match_valide",
                            colonne_groupe="groupe&gvt_affiliation",
                            min_true=50,
                            top_n=10):

    # 1. Déterminer le groupe dominant de chaque orateur (le plus fréquent)
    groupe_dominant = (
        df.groupby([colonne_député, colonne_groupe])
        .size()
        .reset_index(name="nb_mentions")
        .sort_values(["nom_orateur_clean", "nb_mentions"], ascending=[True, False])
        .drop_duplicates(subset=colonne_député)
        .set_index(colonne_député)[colonne_groupe]
    )

    # 2. Compter le nombre total et de "True" pour chaque orateur
    counts = (
        df.groupby(colonne_député)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 3. Calcul de la proportion
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]*100

    # 4. Ajouter la colonne du groupe dominant
    counts[colonne_groupe] = counts[colonne_député].map(groupe_dominant)

    # 5. Ajouter la couleur correspondante
    counts["couleur"] = counts[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes["Autres"])

    # Filtrer les orateurs avec au moins X mentions True
    filtered = counts[counts["true_mentions"] >= min_true]

    # 5. Trier les députés par le nombre de mentions de la FDM "République"
    df_personnelpo_proportion = filtered.sort_values("proportion_true", ascending=False).head(top_n).reset_index(drop=True)


    # 6. Créer le graphique
    fig = go.Figure()

    for _, row in df_personnelpo_proportion.iterrows():
        fig.add_trace(go.Bar(
            x=[row[colonne_député]],
            y=[row["proportion_true"]],
            name=row[colonne_député], #essayer d'enlever
            marker_color=row["couleur"],
            hovertemplate=f"<b>{row[colonne_député]}</b><br>"
                          f"Groupe : {row[colonne_groupe]}<br>"
                          f"Proportions en % : {row['proportion_true']}",
            showlegend=False
        ))

    # 7. Mise en forme
    fig.update_layout(
        title=f"Top {top_n} du personnel politique investissant le plus la famille du mot 'République' à l'AN en proportion (seuil {min_true}) (B)",
        xaxis=dict(title="Député·es -- Ministres", tickangle=-45),
        yaxis=dict(title="Proportion en %", showgrid=False),
        template="plotly_white",
        bargap=0.25,
    )

    fig.show()
    return df_personnelpo_proportion

In [ ]:
df_personnelpo_proportion = diachronique_personnelpo_proportion(df_regroup, couleurs_groupes)
df_personnelpo_proportion

##### En VA et %

In [ ]:
def diachronique_personnelpo_total(df, couleurs_groupes, 
                            colonne_député="nom_orateur_clean",
                            colonne_condition="repu_match_valide",
                            colonne_groupe="groupe&gvt_affiliation",
                            min_true=50,
                            top_n=50, 
                            mode="Double"):
  
    # 1. Déterminer le groupe le plus fréquent pour chaque personnel politique 
    groupe_dominant = (
        df.groupby([colonne_député, colonne_groupe])
        .size()
        .reset_index(name="nb_mentions")
        .sort_values([colonne_député, "nb_mentions"], ascending=[True, False])
        .drop_duplicates(subset=colonne_député)
        .set_index(colonne_député)[colonne_groupe]
    )

    # 2. Compter le nombre d'occurrences de la FDM pour chaque personnel politique
    counts = (
        df.groupby(colonne_député)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 3. Calcul de la proportion (%)
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"] * 100

    # 4. Ajouter le groupe dominant et la couleur
    counts[colonne_groupe] = counts[colonne_député].map(groupe_dominant)
    counts["couleur"] = counts[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes.get("Autres", "grey"))

    # 5. Filtrer le personnel politique à partir d'un minimum d'occurrences
    filtered = counts[counts["true_mentions"] >= min_true]
    df_personnelpo_total = (
        filtered.sort_values("true_mentions", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # 6. Créer la figure Plotly
    fig = go.Figure()

    # --- Mode "absolu" ou "double" : afficher les barres ---
    if mode in ["absolu", "double"]:
        for _, row in df_personnelpo_total.iterrows():
            fig.add_trace(go.Bar(
                x=[row[colonne_député]],
                y=[row["true_mentions"]],
                name=row[colonne_député],
                marker_color=row["couleur"],
                yaxis="y1",
                hovertemplate=f"<b>{row[colonne_député]}</b><br>"
                              f"Groupe : {row[colonne_groupe]}<br>"
                              f"Valeur absolue : {row['true_mentions']}<br>"
                              f"Proportion : {row['proportion_true']:.1f} %",
                showlegend=False
            ))

    # --- Mode "proportion" ou "double" : afficher les points ---
    if mode in ["proportion", "double"]:
        fig.add_trace(go.Scatter(
            x=df_personnelpo_total[colonne_député],
            y=df_personnelpo_total["proportion_true"],
            name="Proportion d'intervention en % avec FDM 'République'",
            mode="markers",
            marker=dict(color="black", size=8, symbol="circle"),
            line=dict(width=1, dash="dot", color="black"),
            yaxis="y2"
        ))

    # 7. Mise en forme du graphique
    fig.update_layout(
        title=(
            f"Top {top_n} du personnel politique investissant le plus la famille du mot 'République' (B) "
            f"(≥ {min_true} occurrences)"
        ),
        xaxis=dict(title="Député·es / Ministres", tickangle=-45),
        yaxis=dict(
            title="Valeur absolue",
            showgrid=False
        ),
        yaxis2=dict(
            title="Proportion en %",
            overlaying="y",
            side="right",
            showgrid=False,
            range=[0, max(df_personnelpo_total["proportion_true"].max() * 1.1, 10)]  # marge auto
        ),
        legend=dict( x=0.6, y=1.2, bgcolor="#E3E1E1"),
        template="plotly_white",
        bargap=0.25
    )

    # 8. Ajustement du titre selon le mode
    if mode == "absolu":
        fig.update_layout(title=fig.layout.title.text + " — en valeur absolue")
    elif mode == "proportion":
        fig.update_layout(title=fig.layout.title.text + " — en proportion (%)")
    else:
        fig.update_layout(title=fig.layout.title.text + "")

    fig.show()

    return df_personnelpo_total


In [ ]:
diachronique_personnelpo_total(df_regroup, couleurs_groupes, mode="double")

### Analyse synchronique

#### En valeur absolue 

In [ ]:
def synchronique_personnel_va(df, couleurs_groupes,
                           date_col="dateSeance_day",
                           colonne_groupe="groupe&gvt_affiliation",
                           colonne_député="nom_orateur_clean",
                           colonne_condition="repu_match_valide",
                           top_n=None):

    # 1. Déterminer le groupe dominant pour chaque personnel politique
    groupe_dominant = (
        df.groupby([colonne_député, colonne_groupe])
        .size()
        .reset_index(name="n")
        .sort_values(["nom_orateur_clean", "n"], ascending=[True, False])
        .drop_duplicates(subset=colonne_député)
        .set_index(colonne_député)[colonne_groupe]
    )

    # 2. Calcul du total de mentions True par personnel politique
    counts = (
        df.groupby(colonne_député)[colonne_condition]
        .apply(lambda x: (x == True).sum())
        .reset_index(name="true_mentions")
    )

    # 3. Sélection des Top N personnel politique
    top_députés = (
        counts.sort_values("true_mentions", ascending=False)
        .head(top_n)[colonne_député]
        .tolist()
    )

    # 4. Filtrer les données correspondantes
    df_top = df[df[colonne_député].isin(top_députés)].copy()

    # 5. Ajouter groupe dominant et couleur
    df_top[colonne_groupe] = df_top[colonne_député].map(groupe_dominant)
    df_top["couleur"] = df_top[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes.get("Autres", "grey"))

    # 6. Agrégation annuelle (YS = début d’année)
    df_agg = (
        df_top.groupby([pd.Grouper(key=date_col, freq="YS"), colonne_député])[colonne_condition]
        .apply(lambda x: (x == True).sum())
        .reset_index(name="true_mentions")
    )

    # 7. Ajouter le groupe et la couleur
    df_agg[colonne_groupe] = df_agg[colonne_député].map(groupe_dominant)
    df_agg["couleur"] = df_agg[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes.get("Autres", "grey"))

    # 8. Liste de symboles pour différencier les députés d’un même groupe
    symboles_possibles = [
        "circle", "square", "diamond", "cross", "triangle-up",
        "triangle-down", "triangle-left", "triangle-right",
        "x", "star", "hexagram", "pentagon"
    ]

    # 9. Attribution automatique des symboles par groupe
    symbol_mapping = {}
    for groupe, sous_df in df_agg.groupby(colonne_groupe):
        députés_groupe = sous_df[colonne_député].unique()
        for i, député in enumerate(députés_groupe):
            symbol_mapping[député] = symboles_possibles[i % len(symboles_possibles)]

    # 10. Création du graphique
    fig = go.Figure()

    for député in top_députés:
        couleur = df_agg.loc[df_agg[colonne_député] == député, "couleur"].iloc[0]
        symbole = symbol_mapping.get(député, "circle")
        df_temp = df_agg[df_agg[colonne_député] == député]

        fig.add_trace(go.Scatter(
            x=df_temp[date_col],
            y=df_temp["true_mentions"],
            mode="lines+markers",
            name=député,
            line=dict(color=couleur, width=1.75),
            marker=dict(size=8, symbol=symbole),
            hovertemplate=(
                f"<b>{député}</b><br>"
                "%{x|%Y}<br>"
                "Occurrences : %{y}"
            )
        ))

    # 11. Mise en forme
    fig.update_layout(
        title=f"Évolution du Top {top_n} du personnel politique investissant le plus la famille du mot 'République' en VA (A)",
        xaxis=dict(title="Année", tickformat="%Y"),
        yaxis=dict(title="Occurrences (valeur absolue)", showgrid=False),
        legend=dict(x=0.85, y=1.15, bgcolor="#E3E1E1"),
        bargap=0.2,
        template="plotly_white"
    )

    fig.show()
    return df_agg


In [ ]:
synchronique_personnel_va(df, couleurs_groupes=couleurs_groupes, top_n=5)


#### En proportion

In [ ]:
def synchronique_personnel_proportion(df, couleurs_groupes,
                           date_col="dateSeance_day",
                           colonne_groupe="groupe&gvt_affiliation",
                           colonne_député="nom_orateur_clean",
                           colonne_condition="repu_match_valide",
                           min_true=None,
                           top_n=None):

    # 1. Déterminer le groupe dominant pour chaque personnel politique
    groupe_dominant = (
        df.groupby([colonne_député, colonne_groupe])
        .size()
        .reset_index(name="n")
        .sort_values([colonne_député, "n"], ascending=[True, False])
        .drop_duplicates(subset=colonne_député)
        .set_index(colonne_député)[colonne_groupe]
    )

    # 2. Calculer total et True par personnel politique
    counts = (
        df.groupby(colonne_député)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 3. Ajouter la proportion globale (%)
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"] * 100

    # 4.  Ajouter le groupe dominant et la couleur
    counts[colonne_groupe] = counts[colonne_député].map(groupe_dominant)
    counts["couleur"] = counts[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes.get("Autres", "grey"))

    # 5. Filtrer les orateurs avec un nombre minimum d’occurrences True
    filtered = counts[counts["true_mentions"] >= min_true]

    # 6. Sélectionner les top N selon la proportion
    top_députés = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[colonne_député]
        .tolist()
    )

    # 7. Filtrer le DF pour ces top N
    df_top = df[df[colonne_député].isin(top_députés)].copy()

    # 8. Agrégation annuelle : total & true
    df_agg = (
        df_top.groupby([pd.Grouper(key=date_col, freq="YS"), colonne_député])[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 9. Ajouter la proportion annuelle (%)
    df_agg["proportion_true"] = df_agg["true_mentions"] / df_agg["total_mentions"] * 100

    # 10. Ajouter le groupe, la couleur et l’année
    df_agg[colonne_groupe] = df_agg[colonne_député].map(groupe_dominant)
    df_agg["couleur"] = df_agg[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes.get("Autres", "grey"))
    df_agg["Année"] = df_agg[date_col].dt.year

    # 11. Définir des symboles distincts pour différencier les personnalités d’un même groupe
    symboles_possibles = [
        "circle", "square", "diamond", "cross", "triangle-up",
        "triangle-down", "triangle-left", "triangle-right",
        "x", "star", "hexagram", "pentagon"
    ]

    symbol_mapping = {}
    for g, sous_df in df_agg.groupby(colonne_groupe):
        députés_groupe = sous_df[colonne_député].unique()
        for i, député in enumerate(députés_groupe):
            symbol_mapping[député] = symboles_possibles[i % len(symboles_possibles)]

    # 12. Création du graphique
    fig = go.Figure()

    for député in top_députés:
        df_temp = df_agg[df_agg[colonne_député] == député]
        couleur = df_temp["couleur"].iloc[0]
        symbole = symbol_mapping.get(député, "circle")

        fig.add_trace(go.Scatter(
            x=df_temp["Année"],
            y=df_temp["proportion_true"],
            mode="lines+markers",
            name=député,
            line=dict(color=couleur, width=2),
            marker=dict(size=8, symbol=symbole),
            hovertemplate=(
                f"<b>{député}</b><br>"
                "Année : %{x}<br>"
                "Proportion : %{y:.1f}%"
            )
        ))

    # 13. Mise en forme finale
    fig.update_layout(
        title=f"Évolution du Top {top_n} du personnel politique investissant la FDM 'République' (en %, seuil {min_true} occurrences)",
    
        xaxis=dict(title="Année", tickformat="%Y"),
        yaxis=dict(title="Proportion d’occurrences (%)", showgrid=False),
        legend=dict(x=0.75, y=1.15, bgcolor="#E3E1E1"),
        template="plotly_white"
    )

    fig.show()
    return df_agg


In [ ]:
synchronique_personnel_proportion(df, couleurs_groupes, top_n=5, min_true=50)


#### En VA et % (trop complexe)

In [ ]:
def synchronique_personnels_proportion(df, couleurs_groupes,
                           date_col="dateSeance_day",
                           colonne_groupe="groupe&gvt_affiliation",
                           colonne_député="nom_orateur_clean",
                           colonne_condition="repu_match_valide",
                           min_true=None,
                           top_n=None):

    # 1. Déterminer le groupe dominant pour chaque personnel politique
    groupe_dominant = (
        df.groupby([colonne_député, colonne_groupe])
        .size()
        .reset_index(name="n")
        .sort_values([colonne_député, "n"], ascending=[True, False])
        .drop_duplicates(subset=colonne_député)
        .set_index(colonne_député)[colonne_groupe]
    )

    # 2. Calculer total et True par personnel politique
    counts = (
        df.groupby(colonne_député)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 3. Ajouter la proportion globale (%)
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"] * 100

    # 4. Ajouter le groupe dominant et la couleur
    counts[colonne_groupe] = counts[colonne_député].map(groupe_dominant)
    counts["couleur"] = counts[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes.get("Autres", "grey"))

    # 5. Filtrer les orateurs avec un nombre minimum d’occurrences True
    filtered = counts[counts["true_mentions"] >= min_true]

    # 6. Sélectionner les top N selon la proportion
    top_députés = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[colonne_député]
        .tolist()
    )

    # 7. Filtrer le DF pour ces top N
    df_top = df[df[colonne_député].isin(top_députés)].copy()

    # 8. Agrégation annuelle : total & true
    df_agg = (
        df_top.groupby([pd.Grouper(key=date_col, freq="YS"), colonne_député])[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 9. Ajouter la proportion annuelle (%)
    df_agg["proportion_true"] = df_agg["true_mentions"] / df_agg["total_mentions"] * 100

    # 10. Ajouter le groupe, la couleur et l’année
    df_agg[colonne_groupe] = df_agg[colonne_député].map(groupe_dominant)
    df_agg["couleur"] = df_agg[colonne_groupe].map(couleurs_groupes).fillna(couleurs_groupes.get("Autres", "grey"))
    df_agg["Année"] = df_agg[date_col].dt.year

    # 11. Symboles distincts pour différencier les personnalités d’un même groupe
    symboles_possibles = [
        "circle", "square", "diamond", "cross", "triangle-up",
        "triangle-down", "triangle-left", "triangle-right",
        "x", "star", "hexagram", "pentagon"
    ]

    symbol_mapping = {}
    for g, sous_df in df_agg.groupby(colonne_groupe):
        députés_groupe = sous_df[colonne_député].unique()
        for i, député in enumerate(députés_groupe):
            symbol_mapping[député] = symboles_possibles[i % len(symboles_possibles)]

    # 12. Création du graphique
    fig = go.Figure()

    for député in top_députés:
        df_temp = df_agg[df_agg[colonne_député] == député]
        couleur = df_temp["couleur"].iloc[0]
        symbole = symbol_mapping.get(député, "circle")

        # Courbe principale : proportion (%)
        fig.add_trace(go.Scatter(
            x=df_temp["Année"],
            y=df_temp["proportion_true"],
            mode="lines+markers",
            name=f"{député} (%)",
            line=dict(color=couleur, width=2),
            marker=dict(size=8, symbol=symbole),
            yaxis="y1",
            hovertemplate=(
                f"<b>{député}</b><br>"
                "Année : %{x}<br>"
                "Proportion : %{y:.1f}%<extra></extra>"
            )
        ))

        # Deuxième trace : valeur absolue (barres fines)
        fig.add_trace(go.Bar(
            x=df_temp["Année"],
            y=df_temp["true_mentions"],
            name=f"{député} (VA)",
            marker_color=couleur,
            opacity=0.5,
            yaxis="y2",
            hovertemplate=(
                f"<b>{député}</b><br>"
                "Année : %{x}<br>"
                "Occurrences : %{y}<extra></extra>"
            ),
            showlegend=False
        ))

    # 13 Mise en forme finale
    fig.update_layout(
        title=f"Évolution annuelle du Top {top_n} du personnel politique investissant le plus la FDM 'République' (seuil {min_true})",
        xaxis=dict(title="Année", tickformat="%Y"),
        yaxis=dict(
            title="Proportion d’occurrences (%)",
            side="right",
            showgrid=False
        ),
        yaxis2=dict(
            title="Valeur absolue (occurrences)",
            overlaying="y",
            side="left",
            showgrid=False
        ),
        legend=dict(x=0.70, y=1.15, bgcolor="#E3E1E1"),
        template="plotly_white",
        bargap=0.25
    )

    fig.show()
    return df_agg


In [ ]:
synchronique_personnels_proportion(
    df,
    couleurs_groupes,
    top_n=5,
    min_true=100
)


### Analyses par personnes 

#### Statistiques 

In [ ]:
def statistiques_personnelpo(df, personnel_po, periode="semaine", colonne_condition="repu_match_valide"):
  
    # Filtrer sur la personne voulue
    df_personnel = df[df["nom_orateur_clean"] == personnel_po].copy()
    if df_personnel.empty:
        raise ValueError(f"Aucune donnée trouvée pour le personnel politique : {personnel_po}")

    # Définir la granularité
    if periode == "semaine":
        df_personnel["periode"] = (
            df_personnel["dateSeance_day"].dt.isocalendar().year.astype(str)
            + "-W"
            + df_personnel["dateSeance_day"].dt.isocalendar().week.astype(str)
        )
    elif periode == "mois":
        df_personnel["periode"] = df_personnel["dateSeance_day"].dt.to_period("M").astype(str)
    elif periode == "annee":
        df_personnel["periode"] = df_personnel["dateSeance_day"].dt.year.astype(str)
    elif periode == "global":
        df_personnel["periode"] = "Global"

    # Compter occurrences par période (proportion et True)
    df_counts = (
        df_personnel.groupby("periode")[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Proportion de True
    df_counts["proportion_true"] = df_counts["true_mentions"] / df_counts["total_mentions"] 

    # Statistiques 
    stats = {
        "moyenne_proportion": df_counts["proportion_true"].mean(),
        "médiane_proportion": df_counts["proportion_true"].median(),
        "moyenne_valeur": df_counts["true_mentions"].mean(),
        "médiane_valeur": df_counts["true_mentions"].median(),
        "deciles_proportion": df_counts["proportion_true"].quantile([i/10 for i in range(0, 10)])*100,
        "deciles_valeur": df_counts["true_mentions"].quantile([i/10 for i in range(0, 10)])
    }
    
    return stats

In [ ]:
personnel_po = "M. Guillaume Larrivé"  

# Sur toute la période 
stats_global = statistiques_personnelpo(df, personnel_po, periode="global")
print(f"Statistiques de {personnel_po} sur 2017-2024")
print(f"Nombre total d'occurrences en VA : {stats_global['moyenne_valeur']:.2f}")
# print(f"Médiane (VA) : {stats_global['médiane_valeur']:.2f}")

# --- Par année ---
stats_annee = statistiques_personnelpo(df, personnel_po, periode="annee")
print(f"Statistiques de {personnel_po} par années")
print(f"Moyenne (VA) : {stats_annee['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_annee['médiane_valeur']:.2f}")
print(f"Moyenne (%)  : {stats_annee['moyenne_proportion']:.2%}")
print(f"Médiane (%)  : {stats_annee['médiane_proportion']:.2%}")
# print(f"Répartition (VA)  : {stats_annee['description_valeur']:}")
# print(f"Répartition (%)  : {stats_annee['description_proportion']:}")
# print(f"Déciles (VA)  : {stats_annee['deciles_valeur']}")
# print(f"Déciles (%)  : {stats_annee['deciles_proportion']}")

# --- Par mois ---
stats_mois = statistiques_personnelpo(df_regroup, personnel_po, periode="mois")
print(f"Statistiques de {personnel_po} par mois")
print(f"Moyenne (VA) : {stats_mois['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_mois['médiane_valeur']:.2f}")
print(f"Moyenne (%)  : {stats_mois['moyenne_proportion']:.2%}")
print(f"Médiane (%)  : {stats_mois['médiane_proportion']:.2%}")
# print(f"Répartition (VA)  : {stats_mois['description_valeur']}")
# print(f"Répartition (%)  : {stats_mois['description_proportion']:}")
print(f"Déciles (VA)  : {stats_mois['deciles_valeur']}")
print(f"Déciles (%)  : {stats_mois['deciles_proportion']}")

# --- Par semaine ---
#stats_sem = statistiques_personnelpo(df, personnel_po, periode="semaine")
#print(f"Statistiques de {personnel_po} par semaines")
#print(f"Moyenne (VA) : {stats_sem['moyenne_valeur']:.2f}")
#print(f"Moyenne (%)  : {stats_sem['moyenne_proportion']:.2%}")
# print(f"Répartition (VA)  : {stats_sem['description_valeur']}")
# print(f"Répartition (%)  : {stats_sem['description_proportion']}")
#print(f"Déciles (VA)  : {stats_sem['deciles_valeur']}")
#print(f"Déciles (%)  : {stats_sem['deciles_proportion']}")


#### Tableau(x) et graphique(s)

In [ ]:
def synchronique_personnel(df, couleurs_groupes, personnel_po, colonne_condition,
    date_col="dateSeance_day",
    colonne_groupe="groupe&gvt_affiliation",
    colonne_personnel="nom_orateur_clean",
    periode="A" 
):
    

    # 1. Filtrer pour la personne choisie
    df = df[df[colonne_personnel] == personnel_po].copy()

    # 2. Déterminer le groupe dominant de la personne
    groupe_dominant = (
        df.groupby(colonne_groupe)
        .size()
        .reset_index(name="n")
        .sort_values("n", ascending=False)
        .iloc[0][colonne_groupe]
    )

    # 3. Rajouter la couleur du groupe/personne (secondaire)
    couleur = couleurs_groupes.get(groupe_dominant, couleurs_groupes.get("Autres", "grey"))

    # 4. Déterminer la fréquence temporelle selon la période choisie
    freq_mapping = {
        "A": ("YS", "Année", "%Y"),
        "M": ("MS", "Mois", "%b %Y"),
        "Q": ("Q", "Trimestre", "T%q %Y")
    }
    if periode not in freq_mapping:
        raise ValueError("periode doit être 'A' (année), 'M' (mois) ou 'Q' (trimestre)")
    
    freq, label_x, tickformat = freq_mapping[periode]

    # 5. Agrégation temporelle 
    df_agg = (
        df.groupby(pd.Grouper(key=date_col, freq=freq))[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 6. Calcul de la proportion et de l’année
    df_agg["proportion_true"] = df_agg["true_mentions"] / df_agg["total_mentions"] * 100
    df_agg["Période"] = df_agg[date_col]
    
    # 7. Création du graphique
    fig = go.Figure()

    # Courbe principale (%)
    fig.add_trace(go.Scatter(
        x=df_agg["Période"],
        y=df_agg["proportion_true"],
        mode="lines+markers",
        name=f"{personnel_po} (%)",
        line=dict(color=couleur, width=3),
        marker=dict(size=8, symbol="circle"),
        yaxis="y1",
        hovertemplate=(
            f"<b>{personnel_po}</b><br>"
            "Année : %{x}<br>"
            "Proportion : %{y:.1f}%<extra></extra>"
        )
    ))

    # Barres = valeurs absolues
    fig.add_trace(go.Bar(
        x=df_agg["Période"],
        y=df_agg["true_mentions"],
        name=f"{personnel_po} (VA)",
        marker_color=couleur,
        opacity=0.4,
        yaxis="y2",
        hovertemplate=(
            f"<b>{personnel_po}</b><br>"
            "Année : %{x}<br>"
            "Occurrences : %{y}<extra></extra>"
        ),
        showlegend=False
    ))

    # 7. Mise en forme finale
    fig.update_layout(
        title=f"Évolution annuelle de l’usage de la FDM 'République' par {personnel_po} ({groupe_dominant})",
        xaxis=dict(title=label_x, tickformat=tickformat),
        yaxis=dict(
            title="Proportion(%)",
            side="right",
            showgrid=False
        ),
        yaxis2=dict(
            title="Valeur absolue",
            overlaying="y",
            side="left",
            showgrid=False
        ),
        legend=dict(x=0.83, y=1.25, bgcolor="#E3E1E1"),
        template="plotly_white",
        bargap=0.25
    )

    fig.show()
    return df_agg


In [ ]:
synchronique_personnel(df,couleurs_groupes,
    colonne_condition="repu_match_valide",
    personnel_po="M. Pierre Meurin",
    periode="M"
)

## Personnel politique par groupes

### Analyse synchronique 

#### Statistiques moyennes

In [ ]:
def statistiques_députés_groupes(df, groupe, periode="semaine", colonne_condition="repu_match_valide"):
  
    # Filtrer sur le parti choisi
    df_groupe = df[df["groupe_députés_affiliation"] == groupe].copy()
    if df_groupe.empty:
        raise ValueError(f"Aucune donnée trouvée pour le groupe : {groupe}")

    # Définir la granularité
    if periode == "semaine":
        df_groupe["periode"] = (
            df_groupe["dateSeance_day"].dt.isocalendar().year.astype(str)
            + "-W"
            + df_groupe["dateSeance_day"].dt.isocalendar().week.astype(str)
        )
    elif periode == "mois":
        df_groupe["periode"] = df_groupe["dateSeance_day"].dt.to_period("M").astype(str)
    elif periode == "annee":
        df_groupe["periode"] = df_groupe["dateSeance_day"].dt.year.astype(str)
    elif periode == "global":
        df_groupe["periode"] = "Global"


    # Compter occurrences par période (proportion et True)
    df_counts = (
        df_groupe.groupby(["nom_orateur_clean", "periode"])[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Proportion de True
    df_counts["proportion_true"] = df_counts["true_mentions"] / df_counts["total_mentions"] 

    # Statistiques 
    stats = {
        "moyenne_proportion": df_counts["proportion_true"].mean(),
        "médiane_proportion": df_counts["proportion_true"].median(),
        "moyenne_valeur": df_counts["true_mentions"].mean(),
        "médiane_valeur": df_counts["true_mentions"].median(),
        "deciles_proportion": df_counts["proportion_true"].quantile([i/100 for i in range(90, 100)])*100,
        "deciles_valeur": df_counts["true_mentions"].quantile([i/100 for i in range(90, 100)])
    }
    
    return stats

In [ ]:
groupe = "LR"  

# Sur toute la période 
stats_global = statistiques_députés_groupes(df, groupe, periode="global")
print("Statistiques sur 2017-2024")
print(f"Moyenne (VA) : {stats_global['moyenne_valeur']:.2f}")
print(f"Médiane (VA) : {stats_global['médiane_valeur']:.2f}")
print(f"Déciles (VA)  : {stats_global['deciles_valeur']}")
print(f"Moyenne (%)  : {stats_global['moyenne_proportion']:.2%}")
print(f"Médiane (%) : {stats_global['médiane_proportion']:.2f}")
print(f"Déciles (%)  : {stats_global['deciles_proportion']}")

# --- Par année ---
# stats_annee = statistiques_députés_groupes(df, groupe, periode="annee")
# print("Par Année :")
# print(f"Moyenne (VA) : {stats_annee['moyenne_valeur']:.2f}")
# print(f"Moyenne (%)  : {stats_annee['moyenne_proportion']:.2%}")
# print(f"Répartition (VA)  : {stats_annee['description_valeur']:}")
# print(f"Répartition (%)  : {stats_annee['description_proportion']:}")
# print(f"Déciles (VA)  : {stats_annee['deciles_valeur']}")
# print(f"Déciles (%)  : {stats_annee['deciles_proportion']}")

# --- Par mois ---
# stats_mois = statistiques_députés_groupes(df, groupe, periode="mois")
#print("Par Mois :")
#print(f"Moyenne (VA) : {stats_mois['moyenne_valeur']:.2f}")
#print(f"Moyenne (%)  : {stats_mois['moyenne_proportion']:.2%}")
#print(f"Répartition (VA)  : {stats_mois['description_valeur']}")
#print(f"Répartition (%)  : {stats_mois['description_proportion']:}")

# --- Par semaine ---
#stats_sem = statistiques_députés_groupes(df, groupe, periode="semaine")
#print("Par Semaine :")
#print(f"Moyenne (VA) : {stats_sem['moyenne_valeur']:.2f}")
#print(f"Moyenne (%)  : {stats_sem['moyenne_proportion']:.2%}")
#print(f"Répartition (VA)  : {stats_sem['description_valeur']}")
#print(f"Répartition (%)  : {stats_sem['description_proportion']}")


#### En valeur absolue

In [ ]:
# Définir les paramètres
groupe = "LR"
colonne_condition = "repu_match_valide"
top_n = 20

# 1. Filtrer pour ne garder que le groupe parlementaire en question 
df_parti = df[df["groupe_députés_affiliation"] == groupe].copy()

# 2. Compter le nombre de fois où chaque orateur du groupe dit "République"
counts = (
    df_parti.groupby("nom_orateur_clean")[colonne_condition]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# 4. Couleur du groupe 
couleur_groupe = couleurs_groupes.get(groupe, couleurs_groupes["Autres"])

# 5. Trier les députés par le nombre de mentions de la FDM "République"
df_counts = counts.sort_values("true_mentions", ascending=False).head(top_n).reset_index(drop=True)

# 6. Créer la figure
fig = go.Figure()

# Barres : volume total 
for _, row in df_counts.iterrows():
    fig.add_trace(go.Bar(
        x=[row["nom_orateur_clean"]],
        y=[row["true_mentions"]],
        name=row["nom_orateur_clean"],
        marker_color=couleur_groupe,
        yaxis="y1",
        showlegend=False  # On affiche la légende séparément si besoin
    ))

# Mise en forme 
fig.update_layout(
    title=f"Top{top_n} des député.es {groupe} investissant le plus la famille du mot 'République' en valeur absolue (A)",
    xaxis=dict(title="Député.es"),
    yaxis=dict(
        title="Valeur absolue",
        showgrid=False
    ),
    template="plotly_white",
    bargap=0.2
)

fig.show()
df_counts


#### En proportion

In [ ]:
# Définir les paramètres
groupe = "LR"
colonne_condition = "repu_match_valide"
min_true = 20
top_n = 20

# 1. Filtrer pour ne garder que le groupe en question 
df_groupe = df[df["groupe_députés_affiliation"] == groupe].copy()

# 2. Compter le nombre de fois où chaque orateur du groupe dit "République"
counts = (
    df_groupe.groupby("nom_orateur_clean")[colonne_condition]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# 3. Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]* 100

# 4. Couleur du groupe 
couleur_groupe = couleurs_groupes.get(groupe, couleurs_groupes["Autres"])

# Filtrer les orateurs avec au moins X mentions True
filtered = counts[counts["true_mentions"] >= min_true]

# 5. Trier les députés par le nombre de mentions de la FDM "République"
df_counts = filtered.sort_values("proportion_true", ascending=False).head(top_n).reset_index(drop=True)

# 6. Créer la figure
fig = go.Figure()

# Barres : volume total 
for _, row in df_counts.iterrows():
    fig.add_trace(go.Bar(
        x=[row["nom_orateur_clean"]],
        y=[row["proportion_true"]],
        name=row["nom_orateur_clean"],
        marker_color=couleur_groupe,
        yaxis="y1",
        showlegend=False  # On affiche la légende séparément si besoin
    ))



# Mise en forme 
fig.update_layout(
    title=f"Top{top_n} des député.es {groupe} investissant le plus la famille du mot 'République' en proportion (A)",
    xaxis=dict(title="Député.es"),
    yaxis=dict(
        title="En %",
        showgrid=False
    ),
    yaxis2=dict(
    ),
    template="plotly_white",
    bargap=0.2
)

fig.show()
df_counts


#### En VA et %

In [ ]:
# Définir les paramètres
groupe = "DEM"
colonne_condition = "repu_match_valide"
top_n = 10

# 1. Filtrer pour ne garder que le groupe en question 
df_groupe = df[df["groupe_députés_affiliation"] == groupe].copy()

# 2. Compter le nombre de fois où chaque orateur du groupe dit "République"
counts = (
    df_groupe.groupby("nom_orateur_clean")[colonne_condition]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# 3. Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]* 100

# 4. Couleur du groupe 
couleur_parti = couleurs_groupes.get(groupe, couleurs_groupes["Autres"])

# 5. Trier les députés par le nombre de mentions de la FDM "République"
df_counts = counts.sort_values("true_mentions", ascending=False).head(top_n).reset_index(drop=True)

# 6. Créer la figure
fig = go.Figure()

# Barres : volume total 
for _, row in df_counts.iterrows():
    fig.add_trace(go.Bar(
        x=[row["nom_orateur_clean"]],
        y=[row["true_mentions"]],
        name=row["nom_orateur_clean"],
        marker_color=couleur_parti,
        yaxis="y1",
        showlegend=False  # On affiche la légende séparément si besoin
    ))

# Points : proportion
fig.add_trace(go.Scatter(
    x=df_counts["nom_orateur_clean"],
    y=df_counts["proportion_true"],
    name="Proportion d'intervention en % avec FDM 'République'",
    mode="markers",
    marker=dict(color="black", size=10, symbol="circle"),
    yaxis="y2"
))

# Mise en forme 
fig.update_layout(
    title=f"Top {top_n} des député.es {groupe} investissant le plus la famille du mot 'République' (A)",
    xaxis=dict(title="Député.es"),
    yaxis=dict(
        title="Valeur absolue",
        showgrid=False
    ),
    yaxis2=dict(
        title="Proportion en %",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(
        x=0.65, y=1.15,
        bgcolor="#E3E1E1"
    ),
    template="plotly_white",
    bargap=0.2
)

fig.show()
df_counts


### Évolutions

#### En valeur absolue

In [ ]:
def evolution_députés_groupes_va(df,
                                  groupe = None, 
                                  date_col="dateSeance_day",
                                  colonne_groupe="groupe_députés_affiliation",
                                  colonne_député="nom_orateur_clean",
                                  colonne_condition="repu_match_valide",
                                  top_n=None):
    
    # 1. Filtrer pour ne garder que le groupe en question 
    df_groupe = df[df[colonne_groupe] == groupe].copy()

    # 2. Calcul des proportions de chaque député par groupe
    counts = (
        df_groupe.groupby(colonne_député)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # 3. Trier les député.s par volume pour ne garder que les X premiers
    top_députés = (
        counts.sort_values("true_mentions", ascending=False)
        .head(top_n)[colonne_député]
        .tolist()
    )
    
    # 4. Filtrer le DF
    df_groupes = df[df[colonne_député].isin(top_députés)&(df[colonne_condition] == True)].copy()

    # 4. Agrégation annuelle 
    df_groupes = (
        df_groupes.groupby([pd.Grouper(key=date_col, freq="YS"), colonne_député])[colonne_condition]
        .apply(lambda x: (x == True).sum())  
        .reset_index(name="true_mentions")
    )

    # 5. Extraire l'année
    df_groupes["Année"] = df_groupes[date_col].dt.year

    # Tracer l'évolution du % dans le temps
    fig = px.line(
            df_groupes,
            x="Année",
            y="true_mentions",
            color=colonne_député,
            markers=True,
            title=f"Évolution du Top {top_n} des député.es {groupe} investissant le plus la famille du mot 'République' (A)",
            labels={
                "true_mentions": "Valeur absolue",
                colonne_député: "Député.s"
            }
        )

    fig.update_layout(
            xaxis=dict(dtick=1),
            hovermode="x unified",
            template="plotly_white",
            legend_title_text="Député.s",
            yaxis_tickformat="",
        )

    fig.show()


In [ ]:
evolution_députés_groupes_va(df, groupe="LR", top_n=5)

#### En proportion (fonction à résoudre)

In [ ]:
def evolution_députés_groupes_pourcentage(df,
                                  groupe = None, 
                                  date_col="dateSeance_day",
                                  colonne_groupe="groupe_députés_affiliation",
                                  colonne_député="nom_orateur_clean",
                                  colonne_condition="repu_match_valide",
                                  top_n=None):
    
    # 1. Filtrer pour ne garder que le groupe en question 
    df_groupe = df[df[colonne_groupe] == groupe].copy()
   
    # 2. Trier les député.s par volume pour ne garder que les X premiers
    top_députés = (
        df_groupe.sort_values("true_mentions", ascending=False)
        .head(top_n)[colonne_député]
        .tolist()
    )
    
    # 3. Filtrer le DF
    df_groupes = df[df[colonne_député].isin(top_députés)&(df[colonne_condition] == True)].copy()

    # 4. Agrégation annuelle 
    df_groupes = (
        df_groupes.groupby([pd.Grouper(key=date_col, freq="YS"), colonne_député])[colonne_condition]
        .apply(lambda x: (x == True).sum())  
        .reset_index()
    )
    
    # 5. Ajouter la proportion annuelle (% d'utilisation)
    df_groupes["proportion_true"] = (df_groupes["true_mentions"] / df_groupes["total_mentions"])
    
    # 6. Extraire l'année
    df_groupes["Année"] = df_groupes[date_col].dt.year

    # Tracer l'évolution du % dans le temps
    fig = px.line(
            df_groupes,
            x="Année",
            y="proportion_true",
            color=colonne_député,
            markers=True,
            title=f"Évolution du Top {top_n} des député.es {groupe} investissant le plus la famille du mot 'République' (A)",
            labels={
                "proportion_true": "En %",
                colonne_député: "Député.s"
            }
        )

    fig.update_layout(
            xaxis=dict(dtick=1),
            hovermode="x unified",
            template="plotly_white",
            legend_title_text="Député.s",
            yaxis_tickformat="",
        )

    fig.show()


In [ ]:
evolution_députés_groupes_pourcentage(df, groupe="LR", top_n=5)

In [ ]:
# TODO : changer manuellement les couleurs des graphiques

def evolution_députés_groupes_pourcentage(df,
                                  groupe=None,
                                  date_col="dateSeance_day",
                                  personnel_col="nom_orateur_clean",
                                  match_col="repu_match_valide",
                                  min_true_mentions=50,
                                  top_n=6):

    # 1. Filtrer pour ne garder que le groupe en question 
    df_groupe = df[df["groupe_députés_affiliation"] == groupe].copy()

    # Calcul global des proportions par groupe
    counts = (
        df_groupe.groupby(personnel_col)[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion globale
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # Filtrer les partis avec au moins min_true_mentions
    filtered = counts[counts["true_mentions"] >= min_true_mentions]

    # Sélectionner les top_n groupes selon la proportion
    top_personnel = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[personnel_col]
        .tolist()
    )

    # Filtrer les données pour ces groupes
    df_top = df[df[personnel_col].isin(top_personnel)].copy()

    # Calculer les stats annuelles : mentions totales et vraies
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="Y"), personnel_col])[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion annuelle (% d'utilisation)
    df_grouped["proportion_true"] = (
        df_grouped["true_mentions"] / df_grouped["total_mentions"]
    )

    # Extraire l'année pour affichage
    df_grouped["Année"] = df_grouped[date_col].dt.year

    # Tracer l'évolution du % dans le temps
    fig = px.line(
        df_grouped,
        x="Année",
        y="proportion_true",
        color=personnel_col,
        markers=True,
        title=f"Évolution du Top {top_n} des député.es {groupe} investissant le plus en proportion la famille du mot 'République' (A)",
        labels={
            "proportion_true": "% d'utilisation",
            personnel_col: "Groupe parlementaire"
        }
    )

    fig.update_layout(
        xaxis=dict(dtick=1),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Député.es",
        yaxis_tickformat=".0%",
    )

    fig.show()


In [ ]:
evolution_députés_groupes_pourcentage(df, groupe="LR")

## Analyse de périodes spécifiques

### Analyse de séquences législatives

#### Quel partis ?

In [28]:
def diachronique_séquences_partis(
    df, 
    couleurs_groupes, 
    jours=None, 
    periode="intervalle", 
    date_debut=None, 
    date_fin=None, 
    seuil_min_true=20, 
    top_n=10,
    date_col="dateSeance_day",
    colonne_condition="repu_match_valide",
    colonne_groupe="groupe&gvt_affiliation",
    titre=None
):

    # 1. Filtrage selon la période 
    if periode == "intervalle":
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df[date_col] >= debut) & (df[date_col] <= fin)]
        titre_defaut = f"Groupes mobilisant le plus la FDM 'République' du {debut.date()} au {fin.date()} (≥ {seuil_min_true} occurrences)"
    elif periode == "jours":
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df[date_col].dt.date.isin(jours_dt)]
        titre_defaut = f"Groupes mobilisant le plus la FDM 'République' lors d'évènements sélectionnés (≥ {seuil_min_true} occurrences)"
    else:
        raise ValueError("La période doit être 'intervalle' ou 'jours'.")

    # 2. Comptage des occurrences 
    counts = (
        df_filtered.groupby(colonne_groupe)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # 3. Filtrage par seuil et tri 
    filtered = counts[counts["true_mentions"] >= seuil_min_true]
    df_top = filtered.sort_values("true_mentions", ascending=False).head(top_n).reset_index(drop=True)

    # 4. Création de la figure 
    fig = go.Figure()

    for _, row in df_top.iterrows():
        groupe = row[colonne_groupe]
        couleur = couleurs_groupes.get(groupe, couleurs_groupes.get("Autres", "grey"))

        fig.add_trace(go.Bar(
            x=[groupe],
            y=[row["true_mentions"]],
            name=f"{groupe} (VA)",
            marker_color=couleur,
            opacity=0.5,
            yaxis="y1",
            hovertemplate="<b>" + groupe + "</b><br>Occurrences : %{y}<extra></extra>",
            showlegend=False
        ))

        fig.add_trace(go.Scatter(
            x=[groupe],
            y=[row["proportion_true"] * 100],
            name=f"{groupe} (%)",
            line=dict(color=couleur, width=3),
            marker=dict(size=10, symbol="circle"),
            yaxis="y2",
            hovertemplate="<b>" + groupe + "</b><br>Proportion : %{y}<extra></extra>"
        ))

    # --- 5. Mise en forme ---
    fig.update_layout(
        title=titre if titre else titre_defaut,
        xaxis=dict(title="Parti / Groupe politique"),
        yaxis=dict(title="Occurrences absolues", showgrid=False),
        yaxis2=dict(
            title="Proportion (%)",
            overlaying="y",
            side="right",
            showgrid=False
        ),
        template="plotly_white",
        legend=dict(x=0.75, y=1.15, bgcolor="rgba(255,255,255,0.7)"),
        bargap=0.4
    )

    fig.show()
    return df_top


##### Projet de loi "séparatisme"

In [29]:
diachronique_séquences_partis(
    df,
    couleurs_groupes=couleurs_groupes,
    date_debut="2021-02-01", date_fin="2021-02-16", 
    periode="intervalle",
    seuil_min_true=10,
    top_n=5
)

,groupe&gvt_affiliation,total_mentions,true_mentions,proportion_true
0,REN,2377,277,0.116533
1,GVT,1643,191,0.116251
2,LR,2167,151,0.069682
3,LFI,783,94,0.120051
4,DEM,780,73,0.093590


In [30]:
diachronique_séquences_partis(
    df,
    couleurs_groupes=couleurs_groupes,
    jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16"], 
    periode="jours",
    seuil_min_true=10,
    top_n=5,
    titre="Groupes mobilisant la FDM de 'République' lors de la 1ère discussion du projet de loi séparatisme"
)


,groupe&gvt_affiliation,total_mentions,true_mentions,proportion_true
0,REN,2178,277,0.127181
1,GVT,1453,186,0.128011
2,LR,1956,150,0.076687
3,LFI,726,94,0.129477
4,DEM,733,73,0.099591


In [31]:
diachronique_séquences_partis(
    df,
    couleurs_groupes=couleurs_groupes,
    jours=["2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"], 
    periode="jours",
    seuil_min_true=10,
    top_n=5,
    titre="Groupes mobilisant la FDM de 'République' lors de la 2e phase de discussion du projet de loi séparatisme"
)


,groupe&gvt_affiliation,total_mentions,true_mentions,proportion_true
0,REN,524,65,0.124046
1,LR,548,63,0.114964
2,LFI,210,44,0.209524
3,GDR,93,22,0.236559
4,SOC-A,88,21,0.238636


In [35]:
diachronique_séquences_partis(
    df_regroup,
    couleurs_groupes=couleurs_groupes,
    jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16", "2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"], 
    periode="jours",
    seuil_min_true=10,
    top_n=5,
    titre="Groupes mobilisant la FDM de 'République' lors de la  discussion du projet de loi séparatisme"
)


,groupe&gvt_affiliation,total_mentions,true_mentions,proportion_true
0,REN,1580,244,0.154430
1,LR,1859,163,0.087682
2,GVT,886,121,0.136569
3,LFI,603,78,0.129353
4,DEM,571,74,0.129597


##### Projet de loi organique « pour une démocratie plus représentative, responsable et efficace »

In [62]:
diachronique_séquences_partis(
    df_regroup,
    couleurs_groupes=couleurs_groupes,
    date_debut="2018-07-10", date_fin="2018-07-22", 
    periode="intervalle",
    seuil_min_true=10,
    top_n=10,
    titre="Groupes mobilisant le plus la FDM de la 'République' (discussion du projet de loi organique démocratie)"
)

,groupe&gvt_affiliation,total_mentions,true_mentions,proportion_true
0,LR,1919,106,0.055237
1,REN,932,64,0.068670
2,LFI,599,62,0.103506
3,GDR,580,57,0.098276
4,SOC-A,564,46,0.081560
5,GVT,545,42,0.077064
6,NI,263,40,0.152091
7,UDI,359,35,0.097493
8,DEM,344,23,0.066860
9,RN,93,11,0.118280


##### Projet de loi "pour une école de la confiance"

In [76]:
diachronique_séquences_partis(
    df,
    couleurs_groupes=couleurs_groupes,
    date_debut="2019-02-11", date_fin="2019-02-19", 
    periode="intervalle",
    seuil_min_true=10,
    top_n=10,
    titre="Groupes mobilisant le plus la FDM de la 'République' lors de la discussion « pour une école de la confiance »"
)

,groupe&gvt_affiliation,total_mentions,true_mentions,proportion_true
0,LR,1377,71,0.051561
1,REN,937,55,0.058698
2,GVT,784,40,0.051020
3,LFI,443,39,0.088036
4,SOC-A,330,21,0.063636
5,LIOT,207,13,0.062802
6,GDR,264,12,0.045455


#### Quelles personnes ?

In [ ]:
def diachronique_séquences_personnel(
    df, 
    couleurs_groupes, 
    jours=None, 
    periode="intervalle", 
    date_debut=None, 
    date_fin=None, 
    seuil_min_true=20, 
    top_n=10,
    date_col="dateSeance_day",
    colonne_condition="repu_match_valide",
    colonne_personnel="nom_orateur_clean",
    colonne_groupe="groupe&gvt_affiliation",
    titre=None
):

    # 1. Filtrage selon la période 
    if periode == "intervalle":
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df[date_col] >= debut) & (df[date_col] <= fin)]
        titre_defaut = f"Personnels mobilisant le plus la FDM 'République' du {debut.date()} au {fin.date()} (≥ {seuil_min_true} occurrences)"
    elif periode == "jours":
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df[date_col].dt.date.isin(jours_dt)]
        titre_defaut = f"Personnels mobilisant le plus la FDM 'République' lors d'évènements sélectionnés (≥ {seuil_min_true} occurrences)"
    else:
        raise ValueError("La période doit être 'intervalle' ou 'jours'.")

    # 2. Comptage des occurrences par personnel
    counts = (
        df_filtered.groupby(colonne_personnel)[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # 3. Ajouter le groupe dominant du personnel pour la couleur
    groupes = (
    df.groupby(colonne_personnel)[colonne_groupe]
    .agg(lambda x: x.value_counts().index[0])  # le groupe le plus fréquent
    .reset_index()
    )
    counts = counts.merge(groupes, on=colonne_personnel, how="left")    

    # 4. Filtrage et tri
    filtered = counts[counts["true_mentions"] >= seuil_min_true]
    df_top = filtered.sort_values("true_mentions", ascending=False).head(top_n).reset_index(drop=True)

    # 5. Création de la figure
    fig = go.Figure()

    for _, row in df_top.iterrows():
        personnel = row[colonne_personnel]
        groupe = row[colonne_groupe]
        couleur = couleurs_groupes.get(groupe, couleurs_groupes.get("Autres", "grey"))

        fig.add_trace(go.Bar(
            x=[personnel],
            y=[row["true_mentions"]],
            name=f"{personnel} (VA)",
            marker_color=couleur,
            opacity=0.5,
            yaxis="y1",
            hovertemplate="<b>" + colonne_personnel + "</b><br>Occurrences : %{y}<extra></extra>",
            showlegend=False
        ))

        fig.add_trace(go.Scatter(
            x=[personnel],
            y=[row["proportion_true"] * 100],
            name=f"{personnel} (%)",
            line=dict(color=couleur, width=3),
            marker=dict(size=10, symbol="circle"),
            yaxis="y2",
            hovertemplate="<b>" + colonne_personnel + "</b><br>Proportion : %{y}<extra></extra>",
            showlegend=False
        ))

    # 6. Mise en forme
    fig.update_layout(
        title=titre if titre else titre_defaut,
        xaxis=dict(title="Personnel politique"),
        yaxis=dict(title="Occurrences absolues", showgrid=False),
        yaxis2=dict(
            title="Proportion (%)",
            overlaying="y",
            side="right",
            showgrid=False
        ),
        template="plotly_white",
        legend=dict(x=0.75, y=1.15, bgcolor="rgba(255,255,255,0.7)"),
        bargap=0.4
    )

    fig.show()
    return df_top

In [63]:
diachronique_séquences_personnel(
    df,
    couleurs_groupes=couleurs_groupes,
    jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16"], 
    periode="jours",
    seuil_min_true=10,
    top_n=10,
    titre="Top 10 du personnel politique sur la FDM de la 'République' lors de la 1ère discussion du projet de loi séparatisme"
)


,nom_orateur_clean,total_mentions,true_mentions,proportion_true,groupe&gvt_affiliation
0,M. Gérald Darmanin,469,103,0.219616,GVT
1,M. Alexis Corbière,355,56,0.157746,LFI
2,Mme Marlène Schiappa,282,46,0.163121,GVT
3,M. Éric Poulliat,162,41,0.253086,REN
4,M. François de Rugy,174,39,0.224138,GVT
5,M. Jean-Christophe Lagarde,150,35,0.233333,UDI
6,M. Xavier Breton,120,28,0.233333,LR
7,M. Florent Boudié,293,26,0.088737,REN
8,M. Boris Vallaud,133,26,0.195489,SOC-A
9,M. Jean-Michel Blanquer,154,25,0.162338,GVT


In [53]:
diachronique_séquences_personnel(
    df,
    couleurs_groupes=couleurs_groupes,
    jours=["2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"], 
    periode="jours",
    seuil_min_true=10,
    top_n=5,
    titre="Top du personnel politique sur la FDM de 'République' lors de la 2e phase de discussion du projet de loi séparatisme"
)


,nom_orateur_clean,total_mentions,true_mentions,proportion_true,groupe&gvt_affiliation
0,M. Alexis Corbière,122,28,0.229508,LFI
1,M. Julien Ravier,54,14,0.259259,LR
2,M. Alain Bruneel,28,12,0.428571,GDR
3,M. Éric Poulliat,42,12,0.285714,REN
4,Mme Lamia El Aaraje,32,11,0.343750,SOC-A


In [55]:
diachronique_séquences_personnel(
    df,
    couleurs_groupes=couleurs_groupes,
    jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16", "2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"], 
    periode="jours",
    seuil_min_true=10,
    top_n=10,
    titre="Top du personnel politique sur la FDM de la 'République' lors de la discussion du projet de loi séparatisme"
)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true,groupe&gvt_affiliation
0,M. Gérald Darmanin,480,104,0.216667,GVT
1,M. Alexis Corbière,477,84,0.176101,LFI
2,M. Éric Poulliat,204,53,0.259804,REN
3,Mme Marlène Schiappa,372,50,0.134409,GVT
4,M. François de Rugy,229,47,0.205240,GVT
5,M. Jean-Christophe Lagarde,150,35,0.233333,UDI
6,M. Florent Boudié,398,34,0.085427,REN
7,M. Xavier Breton,149,30,0.201342,LR
8,M. Jean-Michel Blanquer,174,28,0.160920,GVT
9,M. Julien Ravier,210,27,0.128571,LR


In [72]:
diachronique_séquences_personnel(
    df_regroup,
    couleurs_groupes=couleurs_groupes,
    date_debut="2018-07-10", date_fin="2018-07-22", 
    periode="intervalle",
    seuil_min_true=10,
    top_n=10,
    titre="Top du personnel politique sur la FDM de la 'République' lors de la discussion du projet de loi organique pour une démocratie ..."
)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true,groupe&gvt_affiliation
0,Mme Nicole Belloubet,237,34,0.143460,GVT
1,M. Richard Ferrand,271,29,0.107011,REN
2,M. Sébastien Jumel,235,28,0.119149,GDR
3,M. Jean-Luc Mélenchon,100,22,0.220000,LFI
4,M. Jean-Christophe Lagarde,121,15,0.123967,UDI
5,M. Michel Castellani,59,14,0.237288,LIOT
6,M. Philippe Gosselin,179,14,0.078212,LR
7,M. Jean-Félix Acquaviva,37,11,0.297297,LIOT
8,M. Marc Fesneau,115,11,0.095652,GVT
9,M. Gabriel Serville,28,10,0.357143,GDR


##### Projet de loi "pour une école de la confiance"

In [80]:
diachronique_séquences_personnel(
    df_regroup,
    couleurs_groupes=couleurs_groupes,
    date_debut="2019-02-11", date_fin="2019-02-19", 
    periode="intervalle",
    seuil_min_true=5,
    top_n=10,
    titre="Personnel mobilisant le plus la FDM de la 'République' lors de la discussion « pour une école de la confiance »"
)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true,groupe&gvt_affiliation
0,M. Jean-Michel Blanquer,160,23,0.143750,GVT
1,Mme Anne-Christine Lang,114,14,0.122807,REN
2,M. Alexis Corbière,54,6,0.111111,LFI
3,M. Patrick Hetzel,113,6,0.053097,LR
4,Mme Sabine Rubin,40,6,0.150000,LFI
5,M. Michel Larive,17,5,0.294118,LFI
6,M. Éric Ciotti,15,5,0.333333,LR
7,Mme Danièle Obono,32,5,0.156250,LFI


### Analyses par législatures

In [ ]:
df_16e = df[df["legislature"]== 16]
df_15e = df[df["legislature"]== 15]

# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df_16e.groupby("nom_orateur_clean")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 40]

# Trier par proportion décroissante et garder les 40 premiers
df16_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig16_top20 = px.bar(
    df16_top20,
    x="nom_orateur_clean",
    y="proportion_true",
    title="Top 20 orateurs par proportion de mentions de 'République > 40, 16e législature'",
    labels={"nom_orateur_clean": "Personnel politique", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig16_top20.update_layout(xaxis_tickangle=-45)

fig16_top20.show()
df16_top20

# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df_15e.groupby("nom_orateur_clean")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 100]

# Trier par proportion décroissante et garder les 40 premiers
df15_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig15_top20 = px.bar(
    df15_top20,
    x="nom_orateur_clean",
    y="proportion_true",
    title="Top 20 orateurs par proportion de mentions de 'République > 40 15e législature'",
    labels={"nom_orateur_clean": "Personnel politique", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig15_top20.update_layout(xaxis_tickangle=-45)

fig15_top20.show()
df15_top20

In [ ]:
def top_orateurs_periodique(df, periode="annee", annee=None, semaine=None, jour=None, 
                            date_debut=None, date_fin=None, jours=None,
                            colonne_condition="repu_match_valide",
                            seuil_min_true=None, top_n=20):

    # --- Filtrage selon la période ---
    if periode == "annee":
        if annee is None:
            raise ValueError("Il faut préciser l'année pour periode='annee'")
        df_filtered = df[df["dateSeance_day"].dt.year == annee]
        titre = f"Personnel politique mobilisant en % le plus la 'République' en {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true} occurences)"

    elif periode == "semaine":
        if annee is None or semaine is None:
            raise ValueError("Il faut préciser l'année et la semaine pour periode='semaine'")
        df_filtered = df[
            (df["dateSeance_day"].dt.isocalendar().year == annee) &
            (df["dateSeance_day"].dt.isocalendar().week == semaine)
        ]
        titre = f"Personnel politique mobilisant en % le plus la 'République' la {semaine} semaine {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true})"

    elif periode == "jour":
        if jour is None:
            raise ValueError("Il faut préciser la date pour periode='jour'")
        jour_dt = pd.to_datetime(jour).date()
        df_filtered = df[df["dateSeance_day"].dt.date == jour_dt]
        titre = f"Personnel politique mobilisant le plus en % la 'République' le {jour_dt} (Seuil de {seuil_min_true} occurences)"

    elif periode == "intervalle":
        if date_debut is None or date_fin is None:
            raise ValueError("Il faut préciser date_debut et date_fin pour periode='intervalle'")
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df["dateSeance_day"] >= debut) & (df["dateSeance_day"] <= fin)]
        titre = f"Personnel politique mobilisant le plus en % la 'République' du {debut.date()} au {fin.date()} (Seuil de {seuil_min_true} occurences)"

    elif periode == "jours":
        if jours is None or not isinstance(jours, (list, tuple)):
            raise ValueError("Il faut fournir une liste de dates pour periode='jours'")
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df["dateSeance_day"].dt.date.isin(jours_dt)]
        titre = f"Personnel politique mobilisant le plus la 'République' en % lors de X évènement (Seuil de {seuil_min_true} occurences)"

    else:
        raise ValueError("periode doit être 'annee', 'semaine', 'jour', 'intervalle' ou 'jours'")

    # --- Comptage des occurrences ---
    counts = (
        df_filtered.groupby("nom_orateur_clean")[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # --- Filtrage par seuil ---
    filtered = counts[counts["true_mentions"] >= seuil_min_true]

    # --- Trier par proportion décroissante et garder top_n ---
    df_top = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # --- Graphique ---
    fig = px.bar(
        df_top,
        x="nom_orateur_clean",
        y="proportion_true",
        hover_data=["true_mentions", "total_mentions"],
        title=titre,
        template="plotly_white",
    )
    fig.update_layout(xaxis_tickangle=-45, showlegend=False, yaxis_title="Proportion des interventions totales", xaxis_title="Personnel politique")
    fig.show()

    return df_top


In [ ]:
# Top 10 orateurs en % par année
top_orateurs_periodique(df, periode="annee", annee=2024, seuil_min_true=20)
# Top 10 orateurs sur la semaine X de X
top_orateurs_periodique(df, periode="semaine", annee=2021, semaine=6, seuil_min_true=10)
# Top 10 orateurs en % (tel jour) le X 
top_orateurs_periodique(df, periode="jour", jour="2021-02-01", seuil_min_true=10)

### Quelles évolutions sur la période

In [ ]:
# TODO : changer manuellement les couleurs des graphiques + changer dates pour ressembler plus aux 2 options définies plus haut (intervalle, jours spécifiques)

def evolutions_séquences(df,
                                  date_col="dateSeance_day",
                                  parti_col="groupe_députés_affiliation",
                                  match_col="repu_match_valide",
                                  min_true_mentions=100,
                                  top_n=7,
                                  start_year=2021, # changer les dates
                                  end_year=2021):

    # Filtrer par période si spécifiée
    if start_year or end_year:
        mask = pd.Series(True, index=df.index)
        if start_year:
            mask &= df[date_col].dt.year >= start_year
        if end_year:
            mask &= df[date_col].dt.year <= end_year
        df = df.loc[mask].copy()

    # Calcul global des proportions par groupe
    counts = (
        df.groupby(parti_col)[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # Filtrer les groupes pertinents
    filtered = counts[counts["true_mentions"] >= min_true_mentions]

    # Sélectionner les top groupes selon la proportion
    top_partis = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[parti_col]
        .tolist()
    )

    # Filtrer les données pour ces groupes
    df_top = df[df[parti_col].isin(top_partis)].copy()

    # Calcul annuel des proportions
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="M"), parti_col])[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    df_grouped["proportion_true"] = df_grouped["true_mentions"] / df_grouped["total_mentions"]
    df_grouped["Année"] = df_grouped[date_col].dt.year
    df_grouped["Mois"] = df_grouped[date_col].dt.strftime("%Y-%m")

    # Tracé du graphique (% d’utilisation par année)
    fig = px.line(
        df_grouped,
        x="Mois",
        y="proportion_true",
        color=parti_col,
        markers=True,
        title=(
            f"Évolution mensuelle du % d'utilisation du mot 'République' "
            f"(Des {top_n} principaux groupes parlementaires, {start_year or df_grouped['Année'].min()}–{end_year or df_grouped['Année'].max()})"
        ),
        labels={"proportion_true": "% d'utilisation", parti_col: "Groupe parlementaire"}
    )

    fig.update_layout(
        xaxis=dict(dtick="M1", tickangle=45),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Groupe",
        yaxis_tickformat=".0%"
    )

    fig.show()


In [ ]:
evolutions_séquences(df)

## Autres variables

--> en réalité ce qui serait utile ce serait de faire des vraies stats/régressions pour mesurer le poids de chacune de ces variables sur la probabilité d'utiliser la république. À voir comment faire 

In [ ]:
df["civ"] = df["civ"].replace({"M.": "Homme", "Mme": "Femme"})

In [ ]:
df.to_csv(
    "../data/interim/df_identification_republi_simple.csv",
    index=False,
)

In [ ]:
fig = px.bar(df["civ"].value_counts())
fig.update_layout(
    title="Répartition des genres (civ)", template="plotly_white", showlegend=False
)
fig.show()

### Autres variables 

In [ ]:
# Nécessité ici de transformer l'âge en chiffre pour l'ordonner. 

# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df.groupby("experienceDepute")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_experience_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

df_experience_top20

# Graphique
fig_experience = px.bar(
    df_experience_top20,
    x="experienceDepute",
    y="proportion_true",
    title="Évolution de la proportion d'utilisation de la 'République avec l'expérience",
    labels={"experienceDepute": "Expérience", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_experience.update_layout(xaxis_tickangle=-45)

fig_experience.show()

df_experience_top20

In [ ]:
# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df.groupby("age")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_experienceb_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig_experienceb = px.bar(
    df_experienceb_top20,
    x="age",
    y="proportion_true",
    title="Évolution de la proportion d'utilisation de la 'République avec l'âge",
    labels={"age": "Age", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_experienceb.update_layout(xaxis_tickangle=-45)

fig_experienceb.show()

df_experienceb_top20

In [ ]:
# Compter le nombre de fois où chaque orateur d'un département dit "République"
counts = (
    df.groupby("departementCode")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_dpt_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

df_dpt_top20

# Graphique
fig_dpt = px.bar(
    df_dpt_top20,
    x="departementCode",
    y="proportion_true",
    title="Évolution de la proportion d'utilisation de la 'République par département",
    labels={"departementCode": "Departement", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_dpt.update_layout(xaxis_tickangle=-45)

fig_dpt.show()

df_dpt_top20

==> en VA ce sont les départements IDF et surtout le 93 de LFI qui reviennent le plus, mais en % ce sont les territoires d'outre-mers

In [ ]:
# Compter le nombre de fois où chaque orateur/genre dit "République"
counts = (
    df.groupby("civ")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_genre_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

df_genre_top20

## Test de régressions linéaires 

### Étape 1 : restructuration des variables 

In [ ]:
# Réfléchir à recoder l'âge en génération (reprendre celles de VT ?)

In [ ]:
df["genre"] = df["civ"].replace({"Homme": "1", "Femme": "0"})

In [ ]:
df["repu"] = df["repu_match_valide"].replace({"True": "1", "False": "0"})

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

def regression_lineaire(df, y_col, x_cols):
  
    # Définir X et y
    X = df[x_cols]
    y = df[y_col]
    
    # Encoder les colonnes catégorielles si nécessaire
    X = pd.get_dummies(X, drop_first=True)
    
    # Forcer en 2D si une seule variable explicative
    if X.shape[1] == 1:
        X = X.values.reshape(-1, 1)
    else:
        X = X.values
    
    # y doit être 1D
    y = y.values
    
    # Créer et entraîner le modèle
    model = LinearRegression()
    model.fit(X, y)
    
    # Prédictions
    y_pred = model.predict(X)
    
    # Résultats
    print("Coefficient(s):", model.coef_)
    print("Intercept:", model.intercept_)
    print("Score R²:", model.score(X, y))
    
    # Graphique si une seule variable explicative
    if X.shape[1] == 1:
        plt.scatter(X, y, color="blue", label="Données réelles")
        plt.plot(X, y_pred, color="red", label="Régression")
        plt.xlabel(x_cols[0])
        plt.ylabel(y_col)
        plt.legend()
        plt.show()
    
    return model


In [ ]:
modele = regression_lineaire(df, y_col="repu_match_valide", x_cols=["parti_affiliation"])

In [ ]:
modele = regression_lineaire(df, y_col="repu_match_valide", x_cols=["experienceDepute"])

In [ ]:
modele = regression_lineaire(df, y_col="repu_match_valide", x_cols=["parti_affiliation", "civ", "experienceDepute"])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

def regression_lineaire_df(df, y_col, x_cols, summary=False):
 
    # Définir X (variable à expliquer) et y (variable explicatives)
    X = df[x_cols]
    y = df[y_col]
    
    # Encoder les colonnes catégorielles si nécessaire
    X = pd.get_dummies(X, drop_first=True)
    
    # Forcer en numpy array
    X_values = X.values
    y_values = y.values
    
    # ----- Version scikit-learn -----
    model = LinearRegression()
    model.fit(X_values, y_values)
    y_pred = model.predict(X_values)
    
    print("Régression (scikit-learn)")
    print("Coefficient(s):", model.coef_)
    print("Intercept:", model.intercept_)
    print("Score R²:", model.score(X_values, y_values))
    
    # Graphique si une seule variable explicative
    if X.shape[1] == 1:
        plt.scatter(X_values, y_values, color="blue", label="Données réelles")
        plt.plot(X_values, y_pred, color="red", label="Régression")
        plt.xlabel(x_cols[0])
        plt.ylabel(y_col)
        plt.legend()
        plt.show()
    
    # ----- Version statsmodels -----
    if summary:
        X_sm = sm.add_constant(X)  # ajoute la constante pour l'intercept
        model_sm = sm.OLS(y, X_sm).fit()
        print("Résumé statistique (statsmodels):")
        print(model_sm.summary())
        return model, model_sm
    
    return model


In [ ]:
modele_sklearn, modele_stats = regression_lineaire_df(df, y_col="repu_match_valide", x_cols=["parti_affiliation", "civ"], summary=True)